# Stage 16 — fresh coordinate-moment reader confirmation

This Colab runs exactly one reader-only confirmation on **24 new PushT trajectories**.
It fits on 8 construction trajectories, freezes and hashes every reader, then opens 16
evaluation trajectories and applies one automatic conjunctive gate.

**It does not compute Jacobians, inspect predictor layers, or run causal edits.** The
primary input is the frozen DINOv2 encoding of the true future observation, so a pass
validates the physical measurement map required by later JEPA experiments; it is not
itself evidence about JEPA predictive dynamics.

Run with a GPU using **Runtime → Run all**. Do not edit individual cells. Output is
resumable in Google Drive under `counterfactual_faithfulness_stage16_reader`.

In [ ]:
# SINGLE FROZEN CONFIGURATION BLOCK — do not edit individual cells.
import hashlib
import json

FROZEN_CONFIG = {'ACTIONS_PER_STATE': 13,
 'ACTION_BASIS_DIM': 6,
 'ACTION_PROFILES': 3,
 'ACTION_TANGENT_NORM': 0.35,
 'ACTIVE_CONSTRUCTION_TRAJECTORIES': [100, 103, 106, 109, 112, 115, 118, 121],
 'ACTIVE_EVALUATION_TRAJECTORIES': [101,
                                    102,
                                    104,
                                    105,
                                    107,
                                    108,
                                    110,
                                    111,
                                    113,
                                    114,
                                    116,
                                    117,
                                    119,
                                    120,
                                    122,
                                    123],
 'ACTIVE_HORIZONS': [1, 3],
 'ACTIVE_TIME_INDICES': [0, 1, 2, 3, 4],
 'ASSET_SPECS': {},
 'BOOTSTRAP_SEED': 16111,
 'CONSTRUCTION_TRAJECTORIES': [100, 103, 106, 109, 112, 115, 118, 121],
 'DESIGN_SEED': 200003,
 'DOWNLOAD_RESULTS': True,
 'DRIVE_OUTPUT_DIR': '/content/drive/MyDrive/counterfactual_faithfulness_stage16_reader',
 'ENVIRONMENT': 'PushT',
 'EVALUATION_TRAJECTORIES': [101,
                             102,
                             104,
                             105,
                             107,
                             108,
                             110,
                             111,
                             113,
                             114,
                             116,
                             117,
                             119,
                             120,
                             122,
                             123],
 'EXPECTED_CARRIER_CHANNELS': 400,
 'EXPECTED_CODE_CELL_SHA256': ['43b66627f2fd884e0f4a054438d3464d316a6cfab43598df2af2f4d4d741fc21',
                               'd2c6a45dab4367d6d7d2b0580da2077dec872b598701bc969e8880cabc28269d',
                               'fa6cc69b3550bfbdb3cdd69972eb46d57eed0357e48224e5ddfae32f6772555e',
                               'a6ef88c08c2762015ee87485806d26cee9e38e88ab9c1c2299b318b75a7bf088',
                               '7ff337341ad9975871ca6b3ad95181d9bb49c16ba824ccc4bf045a7f9c984742',
                               'ea9315e100e6e47570fb59f465b8847f4e99c5d92edfb191d1b576d75675a146',
                               '988a6464cd45b40abdc69436e43f2eee612879d92e282d51c57b8069859d655e',
                               'b74d71681c2e75bf608f3dd921d43d3ac6f37ba6aeb542c97cb6734e8d27c591',
                               '0ee8c4a5ba8dcee76bc87745171f8827949fc126c5b37fad027373f4c4c5cb39'],
 'EXPECTED_HF_REVISION': '9b9c41ef249466630dbf1a20e78391865d07b3b9',
 'EXPECTED_PRETRAINED_ASSET_SHA256': {'dinov2_vits14_pretrain.pth': 'b938bf1bc15cd2ec0feacfe3a1bb553fe8ea9ca46a7e1d8d00217f29aef60cd9',
                                      'jepa_wm_pusht.pth.tar': '9beca3eafe0739c3b3adb5d734fa435ccbda0fea8a65d53d4cccec176aaaa0eb'},
 'FIXED_GOAL': [256.0, 256.0, 0.0],
 'FRAMESKIP': 5,
 'FROZEN_PROTOCOL': {'action_bank': {'basis': 'Existing '
                                              'temporal_action_basis(max(horizons)*frameskip, '
                                              'action_profiles=3)',
                                     'basis_dimension': 6,
                                     'branches': 'one zero baseline followed by plus/minus each of '
                                                 'six fixed basis columns',
                                     'maximum_absolute_relative_action': 0.14,
                                     'selection': 'All 13 branches are used. No branch may be '
                                                  'selected or dropped.',
                                     'shape_per_state': [13, 15, 2],
                                     'tangent_norm': 0.35},
                     'claim_boundary': 'A pass validates a physical measurement map in the true '
                                       'future DINOv2 target-representation space. It does not '
                                       'establish JEPA predictive dynamics, internal layer '
                                       'localization, causality, transport, equivariance, object '
                                       'variables, or a vector bundle.',
                     'data_order': ['Materialize and hash all 24 trajectory specifications without '
                                    'rendering or encoding evaluation examples.',
                                    'Generate construction trajectories and true future branches '
                                    'only.',
                                    'Encode construction true target tokens and compute every '
                                    'frozen feature/control matrix.',
                                    'Fit and cross-validate all reader families on construction '
                                    'trajectories only.',
                                    'Serialize reader_freeze.json, all coefficients/normalizers, '
                                    'feature hashes, selected ridge values, protocol hash, and '
                                    'source binding.',
                                    'Verify the evaluation process imports no fitting routine and '
                                    'cannot write reader artifacts.',
                                    'Only then generate and encode evaluation trajectories.',
                                    'Compute target-only variance audit and trajectory-aggregated '
                                    'evaluator metrics.',
                                    'Run stage16_reader_gate.py once. No refitting or alternate '
                                    'report may replace its decision.'],
                     'decision_rule': {'calculator': 'stage16_reader_gate.py',
                                       'conjuncts': ['At each horizon, median six-target skill is '
                                                     'at least 0.30 and every spatial skill is at '
                                                     'least 0.20.',
                                                     'For every deletion of one evaluation '
                                                     'trajectory and at each horizon, median skill '
                                                     'is at least 0.20 and every spatial skill is '
                                                     'strictly positive.',
                                                     'At least 12 of 16 evaluation trajectories '
                                                     'have positive joint two-horizon advantage '
                                                     'over the best matched control.',
                                                     'At each horizon, median relative error '
                                                     'reduction over the best matched control is '
                                                     'at least 0.10.',
                                                     'Every target/horizon passes the frozen '
                                                     'variance-validity rule.',
                                                     'All source, protocol-hash, reader-freeze, '
                                                     'completeness, and forbidden-ID integrity '
                                                     'checks pass.'],
                                       'logic': 'PASS if and only if every conjunct passes. No '
                                                'sensitivity analysis, bootstrap interval, render '
                                                'ceiling, native output, or ordinary R2 can rescue '
                                                'a failure.'},
                     'design_date': '2026-08-03',
                     'estimands': {'best_matched_control': 'For each trajectory and horizon, '
                                                           'minimum six-target mean MSE across '
                                                           'three 1152 CountSketch readers, the '
                                                           'spatial-pool reader, and the '
                                                           'global-scramble reader.',
                                   'joint_advantage': 'For each evaluation trajectory, minimum '
                                                      'over horizons of best_control_MSE minus '
                                                      'primary_degree1_MSE.',
                                   'ordinary_r2': 'Report descriptively for continuity only; never '
                                                  'gate on it.',
                                   'relative_reduction': 'For each trajectory/horizon, '
                                                         '(best_control_MSE - primary_MSE) / '
                                                         'best_control_MSE. A zero control error '
                                                         'yields zero only for zero primary error '
                                                         'and negative infinity otherwise.',
                                   'skill': '1 - aggregate_primary_MSE / '
                                            'aggregate_construction_mean_baseline_MSE',
                                   'trajectory_first_mse': 'Average squared error over 65 examples '
                                                           'within each trajectory and horizon, '
                                                           'then weight trajectories equally.'},
                     'model': {'checkpoint_internal_epoch': 50,
                               'environment': 'PushT',
                               'forbidden_activations': ['predictor block selection',
                                                         'internal carrier selection',
                                                         'post-hoc predicted-token substitution'],
                               'name': 'jepa_wm_pusht',
                               'primary_activation': 'final true future target-encoder visual '
                                                     'tokens',
                               'public_artifact': 'jepa_wm_pusht.pth.tar',
                               'target_encoder': 'DINOv2 ViT-S/14',
                               'target_patch_grid': [16, 16],
                               'target_source': 'MODEL.encode on true future visual and '
                                                'proprioceptive observations; use '
                                                "encoded['visual'][:, :, 0] only",
                               'target_token_shape': [256, 384]},
                     'outcome_actions': {'absolute_fail_specificity_pass': 'Conclude coordinates '
                                                                           'help but the read map '
                                                                           'is too weak for causal '
                                                                           'targeting; stop before '
                                                                           'Jacobians.',
                                         'absolute_pass_specificity_fail': 'Conclude generic '
                                                                           'capacity or coarse '
                                                                           'spatial bias explains '
                                                                           'the gain; stop '
                                                                           'coordinate operators.',
                                         'full_pass': 'Authorize only the prespecified narrow '
                                                      'PushT causal pilot.',
                                         'full_scientific_fail': 'Close the degree-1 direction at '
                                                                 'this checkpoint and preserve all '
                                                                 'negative results.',
                                         'horizon1_only': 'Short-range localization only; reject '
                                                          'longitudinal and transport claims.',
                                         'integrity_fail': 'Pipeline failure. Repair and rerun the '
                                                           'identical protocol without inspecting '
                                                           'model outcomes.',
                                         'stability_or_variance_fail': 'Treat as failed '
                                                                       'confirmation. Any '
                                                                       'redesigned state coverage '
                                                                       'is a new preregistration.'},
                     'patch_coordinates': {'coordinates_sha_requirement': 'Save the exact 256 by 2 '
                                                                          'float64 coordinate '
                                                                          'array and its SHA-256 '
                                                                          'before fitting.',
                                           'image_axis_convention': 'x increases right; y '
                                                                    'increases down',
                                           'token_order': 'row-major 16 by 16 visual patch order',
                                           'x_formula': '2*(column + 0.5)/16 - 1',
                                           'y_formula': '2*(row + 0.5)/16 - 1'},
                     'primary_claim': 'On entirely new PushT trajectories, a prespecified ridge '
                                      'readout of degree-1 coordinate moments from final frozen '
                                      'target-encoder tokens predicts six physical coordinates at '
                                      'horizons 1 and 3, is deletion-stable, and beats dimension-, '
                                      'capacity-, and spatial-bias-matched controls.',
                     'protocol_id': 'stage16-coordinate-moment-confirmation-v1',
                     'protocol_version': 1,
                     'quarantined_prior_data': {'rule': 'May be cited as prior evidence but may '
                                                        'not be loaded by Stage 16 fitting or '
                                                        'evaluation code and may not select a '
                                                        'feature, threshold, layer, horizon, seed, '
                                                        'or hyperparameter.',
                                                'trajectory_ids': [0, 1, 2, 3, 4, 5, 6, 7]},
                     'readers': {'capacity_matched_spatial_pool': {'dimension': 1152,
                                                                   'feature': 'mean each of four '
                                                                              'nonoverlapping 8 by '
                                                                              '8 token quadrants, '
                                                                              'concatenate to '
                                                                              '1536, then apply '
                                                                              'the existing '
                                                                              'norm-stabilized '
                                                                              'CountSketch to 1152 '
                                                                              'dimensions',
                                                                   'projection_generation': 'CountSketchProjector(input_dim=1536, '
                                                                                            'output_dim=1152, '
                                                                                            'seed=16217)',
                                                                   'status': 'matched specificity '
                                                                             'comparator'},
                                 'channel_mean': {'dimension': 384,
                                                  'feature': 'mean(z)',
                                                  'status': 'diagnostic'},
                                 'degree2_sensitivity': {'dimension': 2304,
                                                         'feature': 'concatenate moments for 1, x, '
                                                                    'y, x^2, x*y, y^2',
                                                         'status': 'sensitivity only; cannot '
                                                                   'rescue primary'},
                                 'global_coordinate_scramble': {'dimension': 1152,
                                                                'feature': 'degree-1 moments after '
                                                                           'assigning one fixed '
                                                                           'random permutation of '
                                                                           'the 256 coordinates to '
                                                                           'token positions',
                                                                'seed': 16127,
                                                                'status': 'matched specificity '
                                                                          'comparator'},
                                 'matched_countsketch_1152': {'dimension_each': 1152,
                                                              'gate_aggregation': 'retain three '
                                                                                  'separate '
                                                                                  'readers; use '
                                                                                  'the lowest '
                                                                                  'trajectory MSE '
                                                                                  'as the '
                                                                                  'conservative '
                                                                                  'comparator',
                                                              'implementation': 'Existing '
                                                                                'CountSketchProjector '
                                                                                'on flattened '
                                                                                '256*384 tokens',
                                                              'seeds': [16173, 16187, 16201]},
                                 'native_predicted_proprio': {'input': 'native predictor output if '
                                                                       'exposed by the pinned '
                                                                       'interface',
                                                              'method': 'use the native '
                                                                        'final-horizon vector as a '
                                                                        'separate feature set with '
                                                                        'the same '
                                                                        'construction-only ridge '
                                                                        'procedure',
                                                              'status': 'separate benchmark; '
                                                                        'cannot be concatenated '
                                                                        'with tokens or rescue the '
                                                                        'gate',
                                                              'unavailable_code': 'NATIVE_PROPRIO_UNAVAILABLE'},
                                 'original_countsketch_192': {'dimension': 192,
                                                              'implementation': 'Existing '
                                                                                'CountSketchProjector '
                                                                                'on flattened '
                                                                                '256*384 tokens',
                                                              'prediction_aggregation': 'mean of '
                                                                                        'three '
                                                                                        'frozen '
                                                                                        'reader '
                                                                                        'predictions',
                                                              'ridge_grid': [0.0001,
                                                                             0.001,
                                                                             0.01,
                                                                             0.1,
                                                                             1.0,
                                                                             10.0],
                                                              'seeds': [15161, 15173, 15187],
                                                              'status': 'exact negative-method '
                                                                        'replication; descriptive '
                                                                        'only'},
                                 'primary_degree1': {'dimension': 1152,
                                                     'feature': 'concatenate mean(z), mean(x*z), '
                                                                'mean(y*z)',
                                                     'status': 'sole primary'},
                                 'render_only_oracle': {'input': 'true rendered future image only',
                                                        'method': 'frozen analytic color masks, 16 '
                                                                  'by 16 patch fractions, agent '
                                                                  'centroid plus block centroid '
                                                                  'and central moments, followed '
                                                                  'by the same construction-only '
                                                                  'ridge procedure',
                                                        'status': 'separate measurement ceiling; '
                                                                  'not a model reader and not part '
                                                                  'of the gate'},
                                 'within_example_coordinate_permutation': {'dimension': 1152,
                                                                           'feature': 'degree-1 '
                                                                                      'moments '
                                                                                      'after a '
                                                                                      'label-blind '
                                                                                      'permutation '
                                                                                      'keyed by '
                                                                                      'SeedSequence([16229, '
                                                                                      'trajectory_id, '
                                                                                      'time_index, '
                                                                                      'action_index, '
                                                                                      'horizon])',
                                                                           'status': 'diagnostic; '
                                                                                     'preserves '
                                                                                     'each '
                                                                                     "example's "
                                                                                     'token and '
                                                                                     'coordinate '
                                                                                     'multisets '
                                                                                     'while '
                                                                                     'destroying '
                                                                                     'stable '
                                                                                     'pairing'}},
                     'regression': {'cross_validation': 'leave one construction trajectory out; '
                                                        'minimize equal-target mean of '
                                                        'trajectory-first MSE',
                                    'feature_standardization': 'construction-only mean and '
                                                               'population standard deviation',
                                    'final_fit': 'refit once on all eight construction '
                                                 'trajectories',
                                    'fit_separately_by_horizon': True,
                                    'forbidden': ['output-specific lambda selection',
                                                  'evaluation-dependent clipping',
                                                  'horizon pooling',
                                                  'evaluation-dependent feature removal',
                                                  'nonlinear primary probe'],
                                    'model': 'six-output linear ridge with intercept',
                                    'primary_and_new_control_ridge_grid': [0.0001,
                                                                           0.001,
                                                                           0.01,
                                                                           0.1,
                                                                           1.0,
                                                                           10.0,
                                                                           100.0,
                                                                           1000.0,
                                                                           10000.0,
                                                                           100000.0,
                                                                           1000000.0],
                                    'regularization_selection': 'one common lambda per horizon and '
                                                                'reader family; ties choose the '
                                                                'larger lambda',
                                    'target_standardization': 'construction-only mean and '
                                                              'population standard deviation'},
                     'required_artifacts': ['protocol JSON and SHA-256',
                                            'source_identity.json with embedded '
                                            'config/protocol/code-prefix verification',
                                            'git status/diff and exact commits',
                                            'environment lock and deterministic flags',
                                            'checkpoint and encoder hashes',
                                            'trajectory_design_manifest.json',
                                            'exact patch_coordinates.npy and hash',
                                            'all feature projector/permutation arrays and hashes',
                                            'atomic per-trajectory truth and target-token shards',
                                            'construction CV table',
                                            'reader_freeze.json and frozen coefficient archive',
                                            'target variance audit',
                                            'trajectory-level evaluator_metrics.json',
                                            'stage16_reader_gate.py decision JSON',
                                            'timings, peak GPU memory, logs, and full content '
                                            'manifest'],
                     'resumability': {'evaluation_isolation': 'evaluation loader is unavailable '
                                                              'until reader_freeze.json exists and '
                                                              'its hash is recorded',
                                      'final_evaluator': 'standalone read-only process with no '
                                                         'fitting imports',
                                      'resume_validation': 'shape, dtype, metadata, protocol ID, '
                                                           'source binding, and SHA-256 must all '
                                                           'match',
                                      'write_pattern': 'temporary file, flush/fsync, content hash, '
                                                       'atomic rename'},
                     'runtime_guards': {'abort_rule': 'Abort before evaluation if any guard fails. '
                                                      'Do not loosen a guard after evaluation '
                                                      'artifacts exist.',
                                        'approximate_t4_accelerator_cost_usd_upper': 0.25,
                                        'benchmark_unit': 'one construction trajectory after model '
                                                          'load',
                                        'expected_active_encoding_seconds': 81,
                                        'expected_peak_gpu_gb': 6,
                                        'expected_raw_storage_gb': 1.15,
                                        'expected_wall_minutes': [20, 30],
                                        'maximum_peak_gpu_gb': 12,
                                        'maximum_projected_raw_storage_gb': 3,
                                        'maximum_projected_wall_minutes': 30,
                                        'require_exact_repeat_determinism': True},
                     'sampling': {'actions_per_state': 13,
                                  'construction_count': 8,
                                  'construction_design_indices': [0, 3, 6, 9, 12, 15, 18, 21],
                                  'construction_trajectory_ids': [100,
                                                                  103,
                                                                  106,
                                                                  109,
                                                                  112,
                                                                  115,
                                                                  118,
                                                                  121],
                                  'evaluation_count': 16,
                                  'evaluation_design_indices': [1,
                                                                2,
                                                                4,
                                                                5,
                                                                7,
                                                                8,
                                                                10,
                                                                11,
                                                                13,
                                                                14,
                                                                16,
                                                                17,
                                                                19,
                                                                20,
                                                                22,
                                                                23],
                                  'evaluation_trajectory_ids': [101,
                                                                102,
                                                                104,
                                                                105,
                                                                107,
                                                                108,
                                                                110,
                                                                111,
                                                                113,
                                                                114,
                                                                116,
                                                                117,
                                                                119,
                                                                120,
                                                                122,
                                                                123],
                                  'examples_per_trajectory': 130,
                                  'future_causal_pilot_reserve_if_reader_passes': [101,
                                                                                   104,
                                                                                   107,
                                                                                   110,
                                                                                   113,
                                                                                   116,
                                                                                   119,
                                                                                   122],
                                  'horizons': [1, 3],
                                  'longitudinal_save_steps': [0, 5, 10, 15, 20],
                                  'states_per_trajectory': 5,
                                  'statistical_sampling_unit': 'trajectory',
                                  'task_id_formula': '400 + design_index',
                                  'total_examples': 3120,
                                  'total_trajectories': 24,
                                  'trajectory_id_formula': '100 + design_index'},
                     'scope': 'Reader-only confirmation. No Jacobians, causal edits, layer search, '
                              'model search, or operator shards.',
                     'seeds': {'action_design': 16037,
                               'clustered_bootstrap': 16111,
                               'global_coordinate_scramble': 16127,
                               'master_model_dataloader': 16001,
                               'matched_countsketch_1152': [16173, 16187, 16201],
                               'original_countsketch_192_exact_prior': [15161, 15173, 15187],
                               'spatial_pool_projection': 16217,
                               'split_serialization': 16061,
                               'trajectory_environment_formula': '200003 + 1009 * design_index',
                               'within_example_permutation_root': 16229},
                     'source_contract': {'audited_design_base_commit': '0619cba596cf0efea27d50364f0bbff32b75622c',
                                         'encoder_sha256': 'b938bf1bc15cd2ec0feacfe3a1bb553fe8ea9ca46a7e1d8d00217f29aef60cd9',
                                         'experiment_repository': 'grewalsk/counterfactual-faithfulness-research',
                                         'model_sha256': '9beca3eafe0739c3b3adb5d734fa435ccbda0fea8a65d53d4cccec176aaaa0eb',
                                         'prior_stage15_execution_commit': 'f07a078c34b534bd8a0c5f760af8eba7ccf1bb5c',
                                         'required_before_evaluation': ['Use Runtime > Run all on '
                                                                        'the delivered Stage 16 '
                                                                        'Colab; do not execute an '
                                                                        'edited cell sequence.',
                                                                        'Verify the embedded '
                                                                        'frozen-config hash, '
                                                                        'protocol hash, '
                                                                        'gate-script hash, and '
                                                                        'exact code-cell prefix '
                                                                        'through reader freeze.',
                                                                        'Verify exact repeated '
                                                                        'target encoding before '
                                                                        'reader fitting is '
                                                                        'accepted.',
                                                                        'Write '
                                                                        'source_identity.json and '
                                                                        'refuse evaluation if the '
                                                                        'delivered code prefix '
                                                                        'does not match.'],
                                         'upstream_jepa_wms_commit': '13cf1d9c7e476f53c17714d2e0f1dc239a883ce0'},
                     'status': 'DESIGN_FROZEN_RUN_ONLY_VIA_HASH_VERIFIED_COLAB_ARTIFACT',
                     'targets': {'labels': ['agent_x',
                                            'agent_y',
                                            'block_x',
                                            'block_y',
                                            'block_sin',
                                            'block_cos'],
                                 'orientation': ['sin(theta)', 'cos(theta)'],
                                 'position_normalization': 'divide renderer coordinates by 512',
                                 'spatial_labels': ['agent_x', 'agent_y', 'block_x', 'block_y'],
                                 'standardization': 'horizon-specific construction mean and '
                                                    'population standard deviation only'},
                     'trajectory_generator': {'agent_angle_formula': '0.41 + 2*pi*agent_index/24',
                                              'agent_index_formula': '(5*design_index + 1) mod 24',
                                              'agent_radius_pixels': 125.0,
                                              'analytic_preflight': {'agent_block_x_correlation': -1.1859146676725427e-17,
                                                                     'agent_block_y_correlation': 2.080531125446116e-16,
                                                                     'minimum_initial_agent_block_distance_pixels': 83.25659073448395,
                                                                     'require_absolute_coordinate_correlation_below': 0.05},
                                              'block_angle_formula': '0.17 + 2*pi*design_index/24',
                                              'block_orientation_formula': 'wrap_to_pi(1.3*block_angle)',
                                              'block_radius_pixels': 52.0,
                                              'center_pixels': [256.0, 256.0],
                                              'control_formula': '0.09 * (unit(block-agent) + '
                                                                 'sign(design_index)*0.22*sin(2*pi*step/20)*perpendicular)',
                                              'control_sign_formula': '+1 for even design_index, '
                                                                      '-1 for odd design_index',
                                              'control_steps': 20,
                                              'fixed_goal': [256.0, 256.0, 0.0],
                                              'frameskip': 5,
                                              'initial_velocity_components': 0.0,
                                              'max_relative_action_component': 0.14},
                     'uncertainty': {'bootstrap_role': 'descriptive intervals only',
                                     'clustered_bootstrap_draws': 10000,
                                     'clustered_bootstrap_seed': 16111,
                                     'exact_one_sided_sign_p_at_boundary': 0.0384063720703125,
                                     'exact_specificity_test': 'at least 12 of 16 joint '
                                                               'two-horizon advantages positive',
                                     'power_at_true_positive_probability_0_8': 0.7982454411091969,
                                     'unit': 'trajectory'},
                     'variance_validity': {'calculation': 'Use construction-standardized physical '
                                                          'targets. Compute evaluation pooled SD '
                                                          'around the evaluation mean and '
                                                          'within-trajectory SD over 65 examples '
                                                          'separately for each horizon and target.',
                                           'failure_action': 'Do not drop the coordinate. Mark the '
                                                             'complete confirmation '
                                                             'invalid/failed.',
                                           'minimum_eligible_evaluation_trajectories': 8,
                                           'minimum_pooled_sd_ratio_to_construction': 0.25,
                                           'minimum_within_trajectory_sd_ratio_to_construction': 0.1}},
 'GATE_SCRIPT_SHA256': '6d06b21b2069ff0852c73c68b339219f3910a666a2e4c1b39fbe8e2e3aca4453',
 'GLOBAL_SCRAMBLE_SEED': 16127,
 'HORIZONS': [1, 3],
 'LONGITUDINAL_SAVE_STEPS': [0, 5, 10, 15, 20],
 'MATCHED_SKETCH_SEEDS': [16173, 16187, 16201],
 'MAX_PEAK_GPU_GB': 12.0,
 'MAX_PROJECTED_RAW_GB': 3.0,
 'MAX_PROJECTED_WALL_MINUTES': 30.0,
 'MIN_WITHIN_SD_RATIO': 0.1,
 'MODEL_NAME': 'jepa_wm_pusht',
 'MOUNT_DRIVE': True,
 'ORIGINAL_RIDGE_GRID': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
 'ORIGINAL_SKETCH_SEEDS': [15161, 15173, 15187],
 'OUTPUT_DIR': '/content/counterfactual_faithfulness_stage16_reader',
 'PREDICTOR_BLOCKS': [0, 1, 2, 3, 4, 5],
 'PRE_EVALUATION_CODE_CELL_COUNT': 7,
 'PRIMARY_BLOCK': 3,
 'PRIMARY_HORIZON': 3,
 'PROTOCOL_ID': 'stage16-coordinate-moment-confirmation-v1',
 'PROTOCOL_SHA256': '48eacbf12b86707c5b4d19dee0a2cf661b989a0435a28b8ed4bedfbb74a44e28',
 'READER_LABELS': ['agent_x', 'agent_y', 'block_x', 'block_y', 'block_sin', 'block_cos'],
 'READ_BRANCH': 0,
 'REPO_COMMIT': '13cf1d9c7e476f53c17714d2e0f1dc239a883ce0',
 'REPO_URL': 'https://github.com/facebookresearch/jepa-wms.git',
 'RIDGE_GRID': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0],
 'RUN_MODE': 'confirmation',
 'SEED': 16001,
 'SPATIAL_POOL_SEED': 16217,
 'STATES_PER_TRAJECTORY': 5,
 'TARGET_STEPS': [1, 3],
 'TASK_ID_OFFSET': 400,
 'TOTAL_TRAJECTORIES': 24,
 'WITHIN_PERMUTATION_ROOT': 16229}
FROZEN_CONFIG_SHA256 = '9af408300540b1ef31a2b2f0c282c2f183c1f00c196253e57c91587a479ad8e5'

def digest_json_runtime(value):
    rendered = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(rendered.encode()).hexdigest()

if digest_json_runtime(FROZEN_CONFIG) != FROZEN_CONFIG_SHA256:
    raise RuntimeError("frozen Stage 16 configuration hash mismatch")
if digest_json_runtime(FROZEN_CONFIG["FROZEN_PROTOCOL"]) != FROZEN_CONFIG["PROTOCOL_SHA256"]:
    raise RuntimeError("embedded protocol hash mismatch")
globals().update(FROZEN_CONFIG)
print("Frozen Stage 16 protocol verified:", PROTOCOL_SHA256)

## Frozen decision

The primary reader is degree-1 coordinate moments `[mean(z), mean(xz), mean(yz)]`.
The notebook passes only if both horizons meet the absolute skill thresholds, every
leave-one-trajectory-out subset remains stable, at least 12/16 trajectories beat the
strongest matched control jointly at horizons 1 and 3, the median relative reduction
is at least 10% at each horizon, and every variance/integrity check passes.

In [ ]:
import subprocess
import sys

# Preserve Colab's CUDA-matched torch and torchvision.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
    "scikit-learn==1.6.1",
    "scikit-image==0.24.0",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")

In [ ]:
import csv
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import resource
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as torch_functional
import torchvision
import yaml


def ensure_colab_drive():
    from google.colab import drive
    mountpoint = "/content/drive"
    if not Path(mountpoint, "MyDrive").is_dir():
        drive.mount(mountpoint, timeout_ms=600_000)
    if not Path(mountpoint, "MyDrive").is_dir():
        raise RuntimeError("Google Drive mount did not produce MyDrive")


if MOUNT_DRIVE:
    ensure_colab_drive()
    OUTPUT_DIR = DRIVE_OUTPUT_DIR

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(FROZEN_CONFIG, sort_keys=True, allow_nan=False).encode()
).hexdigest()
OUT = Path(OUTPUT_DIR) / f"confirmation_{RUN_SIGNATURE[:12]}"
ASSET_DIR = OUT / "assets"
DESIGN_DIR = OUT / "design"
TRUTH_DIR = OUT / "truth"
TARGET_DIR = OUT / "target_tokens"
MODEL_DIR = OUT / "frozen_readers"
ANALYSIS_DIR = OUT / "analysis"
EVIDENCE_DIR = OUT / "evaluation_evidence"
PLOT_DIR = OUT / "plots"
LOG_DIR = OUT / "logs"
for directory in [
    OUT, ASSET_DIR, DESIGN_DIR, TRUTH_DIR, TARGET_DIR, MODEL_DIR,
    ANALYSIS_DIR, EVIDENCE_DIR, PLOT_DIR, LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; found {torch.__version__}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.FileHandler(LOG_DIR / "run.log"), logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("stage16_reader")


def write_json(path, payload):
    temporary = Path(path).with_suffix(".tmp.json")
    temporary.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def write_csv(path, rows):
    if not rows:
        return
    temporary = Path(path).with_suffix(".tmp.csv")
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def atomic_npz(path, **arrays):
    temporary = Path(str(path) + ".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_source(value):
    return str(value).replace("\r\n", "\n").strip()


def source_hash(value):
    return hashlib.sha256(canonical_source(value).encode()).hexdigest()


def verify_execution_prefix(expected_count):
    expected = EXPECTED_CODE_CELL_SHA256[:expected_count]
    history = [
        source_hash(value)
        for value in get_ipython().user_ns.get("_ih", [])[1:]
        if canonical_source(value)
    ]
    matched = any(
        history[start : start + len(expected)] == expected
        for start in range(max(0, len(history) - len(expected) + 1))
    )
    identity = {
        "protocol_id": PROTOCOL_ID,
        "protocol_sha256": PROTOCOL_SHA256,
        "frozen_config_sha256": FROZEN_CONFIG_SHA256,
        "expected_code_cells_verified": expected_count,
        "execution_verified": bool(matched),
        "status": (
            "DELIVERED_ARTIFACT_CODE_PREFIX_VERIFIED"
            if matched else "EXECUTED_CODE_PREFIX_MISMATCH"
        ),
        "confirmation_eligible": bool(matched),
    }
    write_json(OUT / "source_identity.json", identity)
    if not matched:
        raise RuntimeError("Restart the runtime and use Runtime > Run all; code-cell history differs from the frozen artifact")
    return identity


VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
}
write_json(OUT / "protocol.json", FROZEN_PROTOCOL)
write_json(OUT / "config.json", {**FROZEN_CONFIG, "run_signature": RUN_SIGNATURE})
write_json(OUT / "versions.json", VERSIONS)
(OUT / "FAILURE_TRACE.txt").write_text("PENDING\n")

TIMINGS = {}
MEMORY = []
PIPELINE_FAILED = False
FAILURE_MESSAGE = ""


def record_failure(stage):
    global PIPELINE_FAILED, FAILURE_MESSAGE
    PIPELINE_FAILED = True
    FAILURE_MESSAGE = f"STAGE: {stage}\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(FAILURE_MESSAGE)
    log.exception("Captured failure in %s", stage)


def memory_report(stage):
    maximum_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    row = {
        "stage": stage,
        "cpu_peak_gib": float(maximum_rss * 1024 / 2**30),
        "gpu_allocated_gib": float(torch.cuda.memory_allocated() / 2**30),
        "gpu_reserved_gib": float(torch.cuda.memory_reserved() / 2**30),
        "gpu_peak_allocated_gib": float(torch.cuda.max_memory_allocated() / 2**30),
    }
    MEMORY.append(row)
    write_json(OUT / "memory.json", MEMORY)
    return row


print(json.dumps(VERSIONS, indent=2))
print(f"Durable run directory: {OUT}")
memory_report("startup")

In [ ]:
def array_sha256(value):
    """Hash an array together with its dtype and shape."""
    array = np.ascontiguousarray(value)
    digest = hashlib.sha256()
    digest.update(str(array.dtype).encode())
    digest.update(str(array.shape).encode())
    digest.update(array.tobytes())
    return digest.hexdigest()


def channel_metric_from_moments(
    count, total, cross, shrinkage=0.10, relative_floor=1e-6
):
    """Fit the same stable metric from streamed first/second moments."""
    count = int(count)
    total = np.asarray(total, dtype=np.float64)
    cross = np.asarray(cross, dtype=np.float64)
    if count < 2 or total.ndim != 1 or cross.shape != (len(total), len(total)):
        raise ValueError("invalid streamed channel moments")
    if not 0.0 <= shrinkage <= 1.0:
        raise ValueError("shrinkage must lie in [0, 1]")
    mean = total / count
    covariance = (cross - np.outer(total, total) / count) / (count - 1)
    covariance = (covariance + covariance.T) / 2.0
    isotropic = np.trace(covariance) / covariance.shape[0]
    covariance = (1.0 - shrinkage) * covariance
    covariance += shrinkage * isotropic * np.eye(covariance.shape[0])
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    maximum = max(float(np.max(eigenvalues)), 1e-12)
    floor = maximum * float(relative_floor)
    eigenvalues = np.maximum(eigenvalues, floor)
    square_root = (eigenvectors * np.sqrt(eigenvalues)[None]) @ eigenvectors.T
    inverse_square_root = (
        eigenvectors * (1.0 / np.sqrt(eigenvalues))[None]
    ) @ eigenvectors.T
    return {
        "mean": mean,
        "covariance": covariance,
        "eigenvalues": eigenvalues,
        "square_root": square_root,
        "inverse_square_root": inverse_square_root,
        "condition_number": float(eigenvalues.max() / eigenvalues.min()),
    }


def transform_primal_channels(values, inverse_square_root):
    """Whiten hidden-space vectors along their final channel dimension."""
    array = np.asarray(values, dtype=np.float64)
    inverse = np.asarray(inverse_square_root, dtype=np.float64)
    if array.shape[-1] != inverse.shape[0] or inverse.shape[0] != inverse.shape[1]:
        raise ValueError("channel metric does not match primal values")
    return np.einsum("...c,dc->...d", array, inverse, optimize=True)


def inverse_transform_primal_channels(values, square_root):
    """Map a whitened hidden-space vector back to native coordinates."""
    array = np.asarray(values, dtype=np.float64)
    root = np.asarray(square_root, dtype=np.float64)
    if array.shape[-1] != root.shape[0] or root.shape[0] != root.shape[1]:
        raise ValueError("channel metric does not match primal values")
    return np.einsum("...c,dc->...d", array, root, optimize=True)


def transform_dual_channels(values, square_root):
    """Transform covectors so dual-primal contractions remain invariant."""
    array = np.asarray(values, dtype=np.float64)
    root = np.asarray(square_root, dtype=np.float64)
    if array.shape[-1] != root.shape[0] or root.shape[0] != root.shape[1]:
        raise ValueError("channel metric does not match dual values")
    return np.einsum("...c,cd->...d", array, root, optimize=True)


def balanced_modes(observability, controllability, tolerance=1e-7):
    """Return empirical balanced primal/dual modes from a small Hankel SVD.

    Parameters
    ----------
    observability:
        Matrix ``G`` with one query-pullback covector per row, shape ``(q,d)``.
    controllability:
        Matrix ``B`` with one action-written direction per column, shape
        ``(d,p)``.

    The construction avoids any ``d x d`` matrix.  If ``H = G B`` and
    ``H = U S V^T``, the returned modes satisfy ``dual.T @ primal ~= I``.
    """
    g = np.asarray(observability, dtype=np.float64)
    b = np.asarray(controllability, dtype=np.float64)
    if g.ndim != 2 or b.ndim != 2 or g.shape[1] != b.shape[0]:
        raise ValueError("observability and controllability shapes disagree")
    hankel = g @ b
    u, singular, vh = np.linalg.svd(hankel, full_matrices=False)
    if not len(singular) or singular[0] <= 0:
        return {
            "hankel": hankel,
            "singular_values": singular,
            "primal": np.empty((b.shape[0], 0), dtype=np.float64),
            "dual": np.empty((b.shape[0], 0), dtype=np.float64),
            "biorthogonality_error": math.nan,
        }
    keep = singular > max(float(singular[0]) * tolerance, 1e-12)
    retained = singular[keep]
    inverse_root = np.diag(1.0 / np.sqrt(retained))
    primal = b @ vh.T[:, keep] @ inverse_root
    dual = g.T @ u[:, keep] @ inverse_root
    identity = np.eye(len(retained), dtype=np.float64)
    error = np.linalg.norm(dual.T @ primal - identity)
    error /= max(np.linalg.norm(identity), 1e-12)
    return {
        "hankel": hankel,
        "singular_values": singular,
        "primal": primal,
        "dual": dual,
        "biorthogonality_error": float(error),
    }


def relative_error(observed, expected):
    """Normalized Frobenius error with a stable zero denominator."""
    left = np.asarray(observed, dtype=np.float64)
    right = np.asarray(expected, dtype=np.float64)
    return float(
        np.linalg.norm(left - right) / max(np.linalg.norm(right), 1e-12)
    )


class CountSketchProjector:
    """Deterministic norm-stabilized projection for context/decoder features."""

    def __init__(self, input_dim, output_dim, seed, device="cuda"):
        rng = np.random.default_rng(int(seed))
        bucket = rng.integers(0, output_dim, size=input_dim, dtype=np.int64)
        sign = rng.choice(np.asarray([-1.0, 1.0], dtype=np.float32), input_dim)
        counts = np.bincount(bucket, minlength=output_dim).astype(np.float32)
        counts[counts == 0] = 1.0
        self.bucket = torch.as_tensor(bucket, device=device, dtype=torch.long)
        self.sign = torch.as_tensor(sign, device=device, dtype=torch.float32)
        self.scale = torch.as_tensor(np.sqrt(counts), device=device)
        self.output_dim = int(output_dim)

    def __call__(self, values):
        values = values.float().flatten(1)
        output = torch.zeros(
            values.shape[0], self.output_dim, device=values.device
        )
        output.scatter_add_(
            1,
            self.bucket[None].expand(values.shape[0], -1),
            values * self.sign[None],
        )
        return output / self.scale[None]


def stable_cosine(left, right):
    left = np.asarray(left, dtype=np.float64).reshape(-1)
    right = np.asarray(right, dtype=np.float64).reshape(-1)
    denominator = np.linalg.norm(left) * np.linalg.norm(right)
    return float(np.dot(left, right) / max(denominator, 1e-12))


def manifest_rows(root, excluded_roots=()):
    excluded = {Path(value).resolve() for value in excluded_roots}
    rows = []
    for path in sorted(Path(root).rglob("*")):
        if not path.is_file() or any(parent in path.resolve().parents for parent in excluded):
            continue
        if path.name.endswith(".part") or path.name.endswith(".tmp"):
            continue
        rows.append(
            {
                "path": str(path.relative_to(root)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )
    return rows


def physical_reader_targets(states):
    """Map PushT states to six fixed, goal-independent physical coordinates."""
    values = np.asarray(states, dtype=np.float64)
    if values.shape[-1] < 5:
        raise ValueError("PushT states must contain at least five coordinates")
    angle = values[..., 4]
    return np.stack(
        [
            values[..., 0] / 512.0,
            values[..., 1] / 512.0,
            values[..., 2] / 512.0,
            values[..., 3] / 512.0,
            np.sin(angle),
            np.cos(angle),
        ],
        axis=-1,
    )


def temporal_action_basis(steps, profiles=3):
    """Return one fixed orthonormal DCT-like action basis for every state.

    Actions are flattened in ``[time, xy]`` order.  The same columns can be
    used at every physical state because they describe perturbations in the
    normalized executable-action coordinates, not state-relative endpoints.
    """
    steps = int(steps)
    profiles = int(profiles)
    if steps < 1 or profiles < 1:
        raise ValueError("steps and profiles must be positive")
    profiles = min(profiles, steps)
    time = np.arange(steps, dtype=np.float64)
    temporal = []
    for frequency in range(profiles):
        column = np.cos(math.pi * (time + 0.5) * frequency / steps)
        column /= np.linalg.norm(column)
        temporal.append(column)
    columns = []
    for column in temporal:
        for axis in range(2):
            value = np.zeros((steps, 2), dtype=np.float64)
            value[:, axis] = column
            columns.append(value.reshape(-1))
    basis = np.stack(columns, axis=1)
    gram = basis.T @ basis
    if not np.allclose(gram, np.eye(len(columns)), atol=1e-12, rtol=0):
        raise AssertionError("temporal action basis is not orthonormal")
    return basis


def fit_ridge(features, targets, ridge):
    """Fit a standardized multi-output ridge model with an explicit intercept."""
    x = np.asarray(features, dtype=np.float64)
    y = np.asarray(targets, dtype=np.float64)
    if x.ndim != 2 or y.ndim != 2 or len(x) != len(y) or len(x) < 2:
        raise ValueError("features and targets must be aligned nontrivial matrices")
    if float(ridge) < 0:
        raise ValueError("ridge must be nonnegative")
    mean = x.mean(axis=0)
    scale = x.std(axis=0)
    scale = np.where(scale > 1e-8, scale, 1.0)
    standardized = (x - mean) / scale
    intercept = y.mean(axis=0)
    centered = y - intercept
    gram = standardized.T @ standardized
    coefficient = np.linalg.solve(
        gram + float(ridge) * np.eye(gram.shape[0]),
        standardized.T @ centered,
    )
    return {
        "feature_mean": mean,
        "feature_scale": scale,
        "intercept": intercept,
        "coefficient": coefficient,
        "ridge": float(ridge),
    }


def predict_ridge(model, features):
    x = np.asarray(features, dtype=np.float64)
    standardized = (
        x - np.asarray(model["feature_mean"], dtype=np.float64)
    ) / np.asarray(model["feature_scale"], dtype=np.float64)
    return (
        standardized @ np.asarray(model["coefficient"], dtype=np.float64)
        + np.asarray(model["intercept"], dtype=np.float64)
    )


def grouped_ridge_cv(features, targets, groups, ridges):
    """Select ridge strength by leave-one-group-out construction-only CV."""
    x = np.asarray(features, dtype=np.float64)
    y = np.asarray(targets, dtype=np.float64)
    group_values = np.asarray(groups)
    ridge_values = tuple(float(value) for value in ridges)
    unique = np.unique(group_values)
    if len(x) != len(y) or len(x) != len(group_values):
        raise ValueError("features, targets, and groups must align")
    if len(unique) < 2 or not ridge_values:
        raise ValueError("grouped CV needs at least two groups and one ridge")
    rows = []
    for ridge in ridge_values:
        losses = []
        for held_out in unique:
            train = group_values != held_out
            validation = ~train
            model = fit_ridge(x[train], y[train], ridge)
            prediction = predict_ridge(model, x[validation])
            loss = float(np.mean((prediction - y[validation]) ** 2))
            losses.append(loss)
            rows.append(
                {
                    "ridge": ridge,
                    "held_out_group": int(held_out),
                    "mse": loss,
                }
            )
        rows.append(
            {
                "ridge": ridge,
                "held_out_group": "mean",
                "mse": float(np.mean(losses)),
            }
        )
    means = {
        row["ridge"]: row["mse"]
        for row in rows
        if row["held_out_group"] == "mean"
    }
    selected = min(ridge_values, key=lambda value: (means[value], value))
    return {
        "model": fit_ridge(x, y, selected),
        "selected_ridge": selected,
        "cv_rows": rows,
    }


def r2_per_output(target, prediction):
    truth = np.asarray(target, dtype=np.float64)
    estimate = np.asarray(prediction, dtype=np.float64)
    if truth.shape != estimate.shape or truth.ndim != 2:
        raise ValueError("target and prediction must be matching matrices")
    residual = np.sum((truth - estimate) ** 2, axis=0)
    centered = np.sum((truth - truth.mean(axis=0)) ** 2, axis=0)
    return 1.0 - residual / np.maximum(centered, 1e-12)


def orthonormal_columns(values, tolerance=1e-9, rank=None):
    """Return a deterministic orthonormal basis for a matrix's column span."""
    matrix = np.asarray(values, dtype=np.float64)
    if matrix.ndim != 2:
        raise ValueError("values must be a matrix")
    if not matrix.size:
        return np.empty((matrix.shape[0], 0), dtype=np.float64)
    u, singular, _ = np.linalg.svd(matrix, full_matrices=False)
    keep = singular > max(float(singular[0]) * float(tolerance), 1e-12)
    available = int(np.sum(keep))
    if rank is not None:
        available = min(available, int(rank))
    return u[:, :available]


def principal_angle_cosines(left, right, rank=None):
    a = orthonormal_columns(left, rank=rank)
    b = orthonormal_columns(right, rank=rank)
    retained = min(a.shape[1], b.shape[1])
    if retained == 0:
        return np.empty(0, dtype=np.float64)
    return np.linalg.svd(a.T @ b, compute_uv=False)[:retained]


def chordal_subspace_distance(left, right, rank=None):
    cosines = principal_angle_cosines(left, right, rank=rank)
    if not len(cosines):
        return math.nan
    return float(np.sqrt(np.mean(np.maximum(1.0 - cosines**2, 0.0))))


def procrustes_align(reference, candidate):
    """Rotate ``candidate`` coordinates to align with ``reference``.

    Both matrices are ambient-by-mode bases with the same number of columns.
    The returned rotation changes only the mode coordinates; it cannot improve
    the underlying subspace overlap.
    """
    target = np.asarray(reference, dtype=np.float64)
    source = np.asarray(candidate, dtype=np.float64)
    if target.ndim != 2 or source.shape != target.shape or not target.shape[1]:
        raise ValueError("reference and candidate must be equal nonempty bases")
    cross = source.T @ target
    u, _, vh = np.linalg.svd(cross, full_matrices=False)
    rotation = u @ vh
    aligned = source @ rotation
    return {
        "aligned": aligned,
        "rotation": rotation,
        "relative_error": float(
            np.linalg.norm(aligned - target)
            / max(np.linalg.norm(target), 1e-12)
        ),
    }


def align_basis_sequence(bases, rank=None):
    """Align a sequence recursively without changing any sampled subspace."""
    if not bases:
        return []
    orthonormal = [orthonormal_columns(value, rank=rank) for value in bases]
    retained = min(value.shape[1] for value in orthonormal)
    if retained == 0:
        return [value[:, :0] for value in orthonormal]
    aligned = [orthonormal[0][:, :retained]]
    for value in orthonormal[1:]:
        result = procrustes_align(aligned[-1], value[:, :retained])
        aligned.append(result["aligned"])
    return aligned


def matrix_cosine(left, right):
    a = np.asarray(left, dtype=np.float64).reshape(-1)
    b = np.asarray(right, dtype=np.float64).reshape(-1)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / max(float(denominator), 1e-12))


def normalized_matrix_distance(left, right):
    a = np.asarray(left, dtype=np.float64)
    b = np.asarray(right, dtype=np.float64)
    if a.shape != b.shape:
        raise ValueError("matrices must have matching shapes")
    return float(
        np.linalg.norm(a - b)
        / max(0.5 * (np.linalg.norm(a) + np.linalg.norm(b)), 1e-12)
    )


def _average_ranks(values):
    array = np.asarray(values, dtype=np.float64)
    order = np.argsort(array, kind="mergesort")
    ranks = np.empty(len(array), dtype=np.float64)
    start = 0
    while start < len(order):
        stop = start + 1
        while stop < len(order) and array[order[stop]] == array[order[start]]:
            stop += 1
        ranks[order[start:stop]] = 0.5 * (start + stop - 1) + 1.0
        start = stop
    return ranks


def spearman_correlation(left, right):
    a = np.asarray(left, dtype=np.float64)
    b = np.asarray(right, dtype=np.float64)
    if a.shape != b.shape or a.ndim != 1 or len(a) < 2:
        raise ValueError("inputs must be aligned nontrivial vectors")
    if not np.all(np.isfinite(a)) or not np.all(np.isfinite(b)):
        raise ValueError("inputs must be finite")
    ra = _average_ranks(a)
    rb = _average_ranks(b)
    ra -= ra.mean()
    rb -= rb.mean()
    denominator = np.linalg.norm(ra) * np.linalg.norm(rb)
    return float(np.dot(ra, rb) / max(float(denominator), 1e-12))


def grouped_label_permutation(labels, groups, seed):
    """Permute time labels independently within each trajectory group."""
    values = np.asarray(labels).copy()
    group_values = np.asarray(groups)
    if len(values) != len(group_values):
        raise ValueError("labels and groups must align")
    rng = np.random.default_rng(int(seed))
    output = values.copy()
    for group in np.unique(group_values):
        indices = np.flatnonzero(group_values == group)
        output[indices] = rng.permutation(values[indices])
    return output


def energy_map(values, tokens=256, channels=400):
    """Return normalized per-token energy for one or more carrier vectors."""
    array = np.asarray(values, dtype=np.float64)
    reshaped = array.reshape(-1, int(tokens), int(channels))
    energy = np.sum(reshaped**2, axis=(0, 2))
    return energy / max(float(np.sum(energy)), 1e-12)


def support_matched_random(template, seed, tokens=256, channels=400):
    """Randomize channel orientation while preserving exact token energies."""
    value = np.asarray(template, dtype=np.float64).reshape(
        int(tokens), int(channels)
    )
    rng = np.random.default_rng(int(seed))
    random = rng.normal(size=value.shape)
    random /= np.maximum(np.linalg.norm(random, axis=1, keepdims=True), 1e-12)
    random *= np.linalg.norm(value, axis=1, keepdims=True)
    result = random.reshape(-1)
    if not np.allclose(
        energy_map(result, tokens=tokens, channels=channels),
        energy_map(value, tokens=tokens, channels=channels),
        atol=1e-12,
        rtol=1e-10,
    ):
        raise AssertionError("support-matched null changed token energies")
    return result


def norm_match(candidate, reference):
    value = np.asarray(candidate, dtype=np.float64)
    target = np.asarray(reference, dtype=np.float64)
    return value * (np.linalg.norm(target) / max(np.linalg.norm(value), 1e-12))


# Pure CPU identities execute before model or simulator access.
_rng = np.random.default_rng(151)
_basis = temporal_action_basis(15, ACTION_PROFILES)
if not np.allclose(_basis.T @ _basis, np.eye(ACTION_BASIS_DIM), atol=1e-12):
    raise AssertionError("fixed action basis CPU identity failed")
_template = _rng.normal(size=256 * EXPECTED_CARRIER_CHANNELS)
_matched = support_matched_random(
    _template, 153, channels=EXPECTED_CARRIER_CHANNELS
)
if not np.allclose(
    energy_map(_template, channels=EXPECTED_CARRIER_CHANNELS),
    energy_map(_matched, channels=EXPECTED_CARRIER_CHANNELS),
):
    raise AssertionError("support-matched null CPU identity failed")

In [ ]:
def to_model_observation(visual, proprio):
    visual_tensor = torch.from_numpy(np.asarray(visual))
    proprio_tensor = torch.from_numpy(np.asarray(proprio))
    if visual_tensor.ndim == 3:
        visual_tensor = visual_tensor[None, None]
        proprio_tensor = proprio_tensor[None, None]
    elif visual_tensor.ndim != 5:
        raise ValueError(f"unexpected visual shape {tuple(visual_tensor.shape)}")
    visual_tensor = visual_tensor.permute(0, 1, 4, 2, 3)
    return {"visual": visual_tensor, "proprio": proprio_tensor}


def configure_repo():
    # Google Drive is reliable for immutable model assets but not as a mutable
    # Git worktree. Use a fresh runtime-local checkout and keep HF/Torch assets
    # in the persistent cache configured by the setup cell.
    repo = Path("/content") / f"stage16-jepa-wms-{REPO_COMMIT[:12]}"
    if repo.exists():
        shutil.rmtree(repo)

    def run_git(command):
        completed = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )
        if completed.returncode != 0:
            raise RuntimeError(
                "git repository setup failed\n"
                f"command: {command!r}\n"
                f"stdout:\n{completed.stdout}\n"
                f"stderr:\n{completed.stderr}"
            )
        return completed

    print(f"Preparing clean ephemeral JEPA-WM source at {repo}")
    run_git(["git", "clone", "--no-checkout", REPO_URL, str(repo)])
    run_git(["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT])
    run_git(["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT])
    resolved = run_git(
        ["git", "-C", str(repo), "rev-parse", "HEAD"]
    ).stdout.strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    # The public hub loader imports a planning entry point with extra evaluation
    # dependencies. Use its equivalent lightweight model constructor.
    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    # Pin the exact Hugging Face snapshot used by the Stage 7 cache.
    hubconf_text = hubconf.read_text()
    filename_marker = 'filename=f"{model_name}.pth.tar",'
    revision_marker = f'revision="{EXPECTED_HF_REVISION}"'
    if revision_marker not in hubconf_text:
        if hubconf_text.count(filename_marker) != 1:
            raise RuntimeError(
                "cannot pin the checkpoint revision in hubconf.py"
            )
        hubconf_text = hubconf_text.replace(
            filename_marker,
            filename_marker
            + f'\n            revision="{EXPECTED_HF_REVISION}",',
        )
        hubconf.write_text(hubconf_text)
    hubconf_text = hubconf.read_text()
    fallback_marker = (
        "        except Exception:\n"
        "            # Fall back to fbaipublicfiles URL\n"
        "            pass\n"
    )
    fail_closed_marker = "pinned Hugging Face checkpoint retrieval failed"
    if fail_closed_marker not in hubconf_text:
        if hubconf_text.count(fallback_marker) != 1:
            raise RuntimeError(
                "cannot disable mutable checkpoint fallback in hubconf.py"
            )
        fallback_replacement = (
            "        except Exception as error:\n"
            "            raise RuntimeError(\n"
            '                "pinned Hugging Face checkpoint retrieval failed"\n'
            "            ) from error\n"
        )
        hubconf_text = hubconf_text.replace(
            fallback_marker,
            fallback_replacement,
        )
        hubconf.write_text(hubconf_text)

    # PushT and Wall do not use the DROID pose helper.
    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in simulator predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    config_paths = [
        repo
        / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/pt/jepa-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/dino-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/jepa-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
    ]
    for config_path in config_paths:
        config = yaml.safe_load(config_path.read_text())
        heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
        heads["architectures"] = {}
        heads["pretrain_dec_path"] = None
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo


def pose_target(environment, states):
    states = np.asarray(states, dtype=np.float64)
    if environment == "PushT":
        angle = states[..., 4]
        return np.stack(
            [
                states[..., 2] / 512.0,
                states[..., 3] / 512.0,
                np.sin(angle),
                np.cos(angle),
            ],
            axis=-1,
        )
    return states[..., :2] / 65.0


def make_environment(repo, environment, task=None):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    if environment == "PushT":
        from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv

        return PushTEnv(
            with_velocity=True,
            with_target=True,
            render_size=224,
            relative=True,
            action_scale=100,
        )

    from evals.simu_env_planning.envs.wall_gym_wrap import DEFAULT_CFG
    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall

    if task is None:
        raise RuntimeError("Stage 14 supports PushT only")
    env = DotWall(
        rng=np.random.default_rng(SEED),
        wall_config=deepcopy(DEFAULT_CFG),
        fix_wall=True,
        cross_wall=False,
        device="cpu",
    )
    env.wall_x = torch.tensor(float(task["wall_x"]))
    env.hole_y = torch.tensor(float(task["door_y"]))
    env.left_wall_x = env.wall_x - env.wall_config.wall_width // 2
    env.right_wall_x = env.wall_x + env.wall_config.wall_width // 2
    return env


def wall_visual(env):
    value = env.render().float()[None]
    resized = torch_functional.interpolate(
        value,
        size=(224, 224),
        mode="bilinear",
        align_corners=False,
        antialias=True,
    )[0]
    return (
        torch.clamp(torch.round(resized), 0, 255)
        .to(torch.uint8)
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )


def reset_environment(repo, environment, task, state, seed):
    if environment == "PushT":
        env = make_environment(repo, environment)
        env.seed(seed)
        env.reset_to_state = np.asarray(state, dtype=np.float64).copy()
        observation, restored = env.reset()
        requested_goal = np.asarray(task["goal"], dtype=np.float64)
        env.set_task_goal(requested_goal.copy())
        if not np.array_equal(np.asarray(env.goal_pose), requested_goal):
            raise RuntimeError("PushT task goal was not installed exactly")
        payload = {
            "visual": np.asarray(env.render("rgb_array")).copy(),
            "proprio": np.asarray(observation["proprio"]).copy(),
        }
        return env, payload, np.asarray(restored).copy()

    env = make_environment(repo, environment, task)
    env.seed(seed)
    env.reset_to_state = torch.as_tensor(
        np.asarray(state, dtype=np.float32)
    )
    observation, restored = env.reset()
    payload = {
        "visual": wall_visual(env),
        "proprio": np.asarray(
            observation["proprio"].detach().cpu(), dtype=np.float32
        ),
    }
    return env, payload, np.asarray(restored.detach().cpu(), dtype=np.float32)


def rollout_branch(repo, environment, task, state, actions, seed):
    env, initial, restored = reset_environment(
        repo, environment, task, state, seed
    )
    wanted = set(TARGET_STEPS)
    observations = {}
    states = {}
    interactions = {}
    interaction_types = {}
    cumulative_interactions = 0
    cumulative_crossings = 0
    previous_state = np.asarray(restored, dtype=np.float64).copy()

    for step, action in enumerate(actions, start=1):
        if environment == "PushT":
            observation, _, _, info = env.step(action)
            current_state = np.asarray(info["state"]).copy()
            cumulative_interactions += int(info.get("n_contacts", 0))
            current_observation = {
                "visual": np.asarray(observation["visual"]).copy(),
                "proprio": np.asarray(observation["proprio"]).copy(),
            }
        else:
            observation, _, _, info = env.step(
                torch.as_tensor(action, dtype=torch.float32)
            )
            current_state = np.asarray(
                info["state"].detach().cpu(), dtype=np.float64
            )
            proposed = previous_state + 2.0 * np.asarray(action)
            if np.linalg.norm(current_state - proposed) > 1e-5:
                cumulative_interactions += 1
            wall_x = float(task["wall_x"])
            crossed = (
                (previous_state[0] - wall_x) * (current_state[0] - wall_x)
                < 0
            )
            cumulative_crossings += int(crossed)
            current_observation = {
                "visual": wall_visual(env),
                "proprio": np.asarray(
                    observation["proprio"].detach().cpu(),
                    dtype=np.float32,
                ),
            }
        previous_state = current_state.copy()
        if step % FRAMESKIP == 0:
            horizon = step // FRAMESKIP
            if horizon in wanted:
                observations[horizon] = current_observation
                states[horizon] = current_state
                interactions[horizon] = cumulative_interactions
                if environment == "PushT":
                    interaction_types[horizon] = (
                        "contact" if cumulative_interactions > 0 else "free"
                    )
                elif cumulative_interactions > 0:
                    interaction_types[horizon] = "collision"
                elif cumulative_crossings > 0:
                    interaction_types[horizon] = "door_cross"
                else:
                    interaction_types[horizon] = "free"
    if wanted != set(observations):
        raise RuntimeError(f"missing horizons: {wanted - set(observations)}")
    return (
        initial,
        restored,
        observations,
        states,
        interactions,
        interaction_types,
    )


def exact_restore_test(repo, environment, task, state, actions):
    endpoints = []
    images = []
    interactions = []
    for _ in range(3):
        initial, _, _, states, counts, kinds = rollout_branch(
            repo,
            environment,
            task,
            state,
            actions,
            SEED + 9000,
        )
        endpoints.append(states[max(HORIZONS)])
        images.append(initial["visual"])
        interactions.append(
            (counts[max(HORIZONS)], kinds[max(HORIZONS)])
        )
    result = {
        "environment": environment,
        "repeats": 3,
        "endpoint_bitwise_exact": all(
            np.array_equal(endpoints[0], item) for item in endpoints[1:]
        ),
        "initial_render_bitwise_exact": all(
            np.array_equal(images[0], item) for item in images[1:]
        ),
        "diagnostics_exact": all(
            interactions[0] == item for item in interactions[1:]
        ),
        "max_endpoint_abs_diff": float(
            max(
                np.max(np.abs(endpoints[0] - item))
                for item in endpoints[1:]
            )
        ),
    }
    if not all(
        result[key]
        for key in [
            "endpoint_bitwise_exact",
            "initial_render_bitwise_exact",
            "diagnostics_exact",
        ]
    ):
        raise AssertionError(f"exact restoration failed: {result}")
    return result


def verify_pretrained_assets():
    rows = []
    for name, expected in EXPECTED_PRETRAINED_ASSET_SHA256.items():
        matching = [
            path for path in CACHE_ROOT.rglob(name)
            if sha256_file(path) == expected
        ]
        if not matching:
            raise RuntimeError(f"verified pretrained asset not found: {name}")
        rows.append({"name": name, "path": str(matching[0]), "sha256": expected})
    write_json(OUT / "pretrained_asset_verification.json", rows)
    return rows


def validate_jepa_predictor(model):
    predictor = model.model.predictor
    blocks = list(getattr(predictor, "predictor_blocks", []))
    if predictor.__class__.__name__ != "VisionTransformerAdaLN":
        raise RuntimeError(f"unexpected predictor class {predictor.__class__.__name__}")
    if len(blocks) != 6:
        raise RuntimeError(f"expected six predictor blocks, found {len(blocks)}")
    if not bool(getattr(predictor, "action_encoder_inpred", False)):
        raise RuntimeError("predictor does not internally encode actions")
    if bool(getattr(predictor, "use_activation_checkpointing", False)):
        raise RuntimeError("activation checkpointing must be disabled for hooks/JVPs")
    if int(getattr(model, "ctxt_window", -1)) != 2:
        raise RuntimeError(f"expected context window 2, found {model.ctxt_window}")
    return predictor, blocks


def load_frozen_model():
    model, preprocessor = torch.hub.load(
        str(REPO),
        MODEL_NAME,
        source="local",
        pretrained=True,
        device="cuda:0",
        trust_repo=True,
    )
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    predictor, blocks = validate_jepa_predictor(model)
    verify_pretrained_assets()
    return model, preprocessor, predictor, blocks


def model_action_tensor(preprocessor, selected_actions, horizon):
    selected_actions = np.asarray(selected_actions, dtype=np.float32)
    chunks = torch.from_numpy(
        selected_actions[:, : horizon * FRAMESKIP].reshape(
            ACTIONS_PER_STATE, horizon, FRAMESKIP, 2
        )
    ).float()
    normalized = preprocessor.normalize_actions(chunks)
    return (
        normalized.reshape(ACTIONS_PER_STATE, horizon, -1)
        .permute(1, 0, 2)
        .contiguous()
        .cuda()
    )


def layer_tokens_full(capture):
    if capture.ndim != 3 or capture.shape[1] % 256:
        raise ValueError(f"unexpected block output {tuple(capture.shape)}")
    if capture.shape[-1] != EXPECTED_CARRIER_CHANNELS:
        raise ValueError(
            f"expected carrier width {EXPECTED_CARRIER_CHANNELS}, "
            f"found {capture.shape[-1]}"
        )
    return capture.view(
        capture.shape[0], capture.shape[1] // 256, 256, capture.shape[-1]
    )[:, -1]


def forward_with_carriers(
    initial,
    actions,
    horizon,
    capture_blocks=(),
    intervention=None,
    require_grad=False,
):
    captures = {int(block): [] for block in capture_blocks}
    context = {"step": -1}
    handles = []
    for block_index in capture_blocks:
        def hook(_module, _inputs, output, block_index=int(block_index)):
            # With frozen parameters and ordinary inputs, autograd would not
            # create a graph. Anchor the requested carrier as a leaf so the
            # suffix VJP is exact without retaining the upstream encoder.
            if require_grad and not output.requires_grad:
                output = output.detach().requires_grad_(True)
            captures[block_index].append(output)
            if (
                intervention is None
                or block_index != int(intervention["block"])
                or context["step"] != horizon - 1
            ):
                return output
            value = output.clone()
            view = value.view(
                value.shape[0], value.shape[1] // 256, 256, value.shape[-1]
            )
            replacement = intervention.get("replacement")
            if replacement is not None:
                view[:, -1] = replacement.to(view.device, view.dtype)
            else:
                delta = intervention["delta"].to(view.device, view.dtype)
                view[:, -1] = view[:, -1] + delta
            return view.reshape_as(value)

        handles.append(PREDICTOR_BLOCK_MODULES[block_index].register_forward_hook(hook))

    try:
        batch = actions.shape[1]
        action_batch = actions[:horizon].permute(1, 0, 2).contiguous()
        with torch.set_grad_enabled(require_grad):
            action_features = MODEL.model.encode_act(action_batch)
            if action_features.shape[-1] != 10:
                raise RuntimeError(
                    f"expected encoded action width 10, found {action_features.shape[-1]}"
                )
            visual_history = initial["visual"].expand(
                batch, *initial["visual"].shape[1:]
            ).detach().clone()
            proprio_history = initial["proprio"].expand(
                batch, *initial["proprio"].shape[1:]
            ).detach().clone()
            predicted_tokens = None
            predicted_proprio = None
            for step_index in range(horizon):
                context["step"] = step_index
                predicted_visual, _, predicted_proprio = MODEL.model.forward_pred(
                    visual_history[:, -MODEL.ctxt_window :],
                    action_features[:, : step_index + 1][:, -MODEL.ctxt_window :],
                    proprio_history[:, -MODEL.ctxt_window :],
                )
                next_visual = predicted_visual[:, -1:]
                next_proprio = predicted_proprio[:, -1:]
                predicted_tokens = next_visual[:, 0, 0].flatten(1, 2)
                if predicted_tokens.shape[1:] != (256, 384):
                    raise RuntimeError(
                        f"expected visual grid [256,384], found {predicted_tokens.shape[1:]}"
                    )
                visual_history = torch.cat([visual_history, next_visual], dim=1)
                proprio_history = torch.cat([proprio_history, next_proprio], dim=1)
        final_captures = {
            block: captures[block][-1] for block in capture_blocks
        }
        return predicted_tokens, predicted_proprio[:, -1], final_captures
    finally:
        for handle in handles:
            handle.remove()

In [ ]:
# Freeze all 24 trajectory specifications before rendering any evaluation data.


def trajectory_specs():
    specs = []
    center = np.asarray([256.0, 256.0])
    for design_index in range(TOTAL_TRAJECTORIES):
        trajectory_id = 100 + design_index
        block_angle = 0.17 + 2.0 * np.pi * design_index / TOTAL_TRAJECTORIES
        block = center + 52.0 * np.asarray([np.cos(block_angle), np.sin(block_angle)])
        agent_index = (5 * design_index + 1) % TOTAL_TRAJECTORIES
        agent_angle = 0.41 + 2.0 * np.pi * agent_index / TOTAL_TRAJECTORIES
        agent = center + 125.0 * np.asarray([np.cos(agent_angle), np.sin(agent_angle)])
        direction = block - agent
        direction /= max(np.linalg.norm(direction), 1e-12)
        perpendicular = np.asarray([-direction[1], direction[0]])
        controls = []
        for step in range(1, LONGITUDINAL_SAVE_STEPS[-1] + 1):
            fraction = step / LONGITUDINAL_SAVE_STEPS[-1]
            curve = (
                (1.0 if design_index % 2 == 0 else -1.0)
                * 0.22 * np.sin(2.0 * np.pi * fraction) * perpendicular
            )
            action = 0.09 * (direction + curve)
            if np.linalg.norm(action) > 0.14:
                raise RuntimeError("longitudinal relative action is too large")
            controls.append(action)
        split = "construction" if trajectory_id in CONSTRUCTION_TRAJECTORIES else "evaluation"
        specs.append({
            "design_index": design_index,
            "trajectory_id": trajectory_id,
            "task_id": 400 + design_index,
            "split": split,
            "seed": 200003 + 1009 * design_index,
            "initial_state": np.asarray([
                agent[0], agent[1], block[0], block[1],
                ((1.3 * block_angle + np.pi) % (2 * np.pi)) - np.pi,
                0.0, 0.0, 0.0, 0.0, 0.0,
            ], dtype=np.float64),
            "controls": np.asarray(controls, dtype=np.float64),
        })
    return specs


ALL_TRAJECTORY_SPECS = trajectory_specs()
CONSTRUCTION_SPECS = [row for row in ALL_TRAJECTORY_SPECS if row["trajectory_id"] in CONSTRUCTION_TRAJECTORIES]
EVALUATION_SPECS = [row for row in ALL_TRAJECTORY_SPECS if row["trajectory_id"] in EVALUATION_TRAJECTORIES]
if [row["trajectory_id"] for row in CONSTRUCTION_SPECS] != CONSTRUCTION_TRAJECTORIES:
    raise AssertionError("construction split drifted")
if [row["trajectory_id"] for row in EVALUATION_SPECS] != EVALUATION_TRAJECTORIES:
    raise AssertionError("evaluation split drifted")
if {row["trajectory_id"] for row in ALL_TRAJECTORY_SPECS} & set(range(8)):
    raise AssertionError("quarantined Stage 15 trajectory ID entered Stage 16")

initials = np.stack([row["initial_state"] for row in ALL_TRAJECTORY_SPECS])
corr_x = float(np.corrcoef(initials[:, 0], initials[:, 2])[0, 1])
corr_y = float(np.corrcoef(initials[:, 1], initials[:, 3])[0, 1])
if abs(corr_x) >= 0.05 or abs(corr_y) >= 0.05:
    raise AssertionError("agent/block coordinate sampling is coupled")

atomic_npz(
    DESIGN_DIR / "stage16_design.npz",
    design_indices=np.asarray([row["design_index"] for row in ALL_TRAJECTORY_SPECS]),
    trajectory_ids=np.asarray([row["trajectory_id"] for row in ALL_TRAJECTORY_SPECS]),
    splits=np.asarray([row["split"] for row in ALL_TRAJECTORY_SPECS]),
    initial_states=initials,
    controls=np.stack([row["controls"] for row in ALL_TRAJECTORY_SPECS]),
    save_steps=np.asarray(LONGITUDINAL_SAVE_STEPS),
)
write_json(DESIGN_DIR / "trajectory_design_manifest.json", {
    "protocol_id": PROTOCOL_ID,
    "specs": [{
        **{key: value for key, value in row.items() if key not in {"initial_state", "controls"}},
        "initial_state": row["initial_state"].tolist(),
        "controls": row["controls"].tolist(),
    } for row in ALL_TRAJECTORY_SPECS],
    "construction_trajectories": CONSTRUCTION_TRAJECTORIES,
    "evaluation_trajectories": EVALUATION_TRAJECTORIES,
    "save_steps": LONGITUDINAL_SAVE_STEPS,
    "horizons": HORIZONS,
    "agent_block_x_correlation": corr_x,
    "agent_block_y_correlation": corr_y,
})
print(f"Frozen {len(CONSTRUCTION_SPECS)} construction and {len(EVALUATION_SPECS)} evaluation trajectories")

In [ ]:
# Realize construction trajectories and exact future-action branches only.


def record_task(record_or_spec):
    return {
        "environment": ENVIRONMENT,
        "task_id": int(record_or_spec["task_id"]),
        "goal": list(FIXED_GOAL),
    }


def dynamic_state_from_environment(environment):
    return np.asarray(
        [
            *environment.agent.position,
            *environment.block.position,
            float(environment.block.angle),
            *environment.agent.velocity,
            *environment.block.velocity,
            float(environment.block.angular_velocity),
        ],
        dtype=np.float64,
    )


def reset_dynamic_environment(dynamic_state, task, seed):
    state = np.asarray(dynamic_state, dtype=np.float64)
    if state.shape != (10,):
        raise ValueError(f"expected ten-dimensional dynamic state, found {state.shape}")
    environment = make_environment(REPO, ENVIRONMENT)
    environment.seed(int(seed))
    environment.reset_to_state = np.asarray(
        [*state[:5], 0.0, 0.0], dtype=np.float64
    )
    environment.reset()
    environment.agent.position = tuple(state[:2])
    environment.block.angle = float(state[4])
    environment.block.position = tuple(state[2:4])
    environment.agent.velocity = tuple(state[5:7])
    environment.block.velocity = tuple(state[7:9])
    environment.block.angular_velocity = float(state[9])
    environment.set_task_goal(np.asarray(task["goal"], dtype=np.float64))
    restored = dynamic_state_from_environment(environment)
    if not np.allclose(restored, state, atol=1e-12, rtol=0):
        raise RuntimeError(
            f"full dynamic restoration drifted: {np.max(np.abs(restored - state))}"
        )
    observation = {
        "visual": np.asarray(environment.render("rgb_array")).copy(),
        "proprio": np.asarray(
            [*environment.agent.position, *environment.agent.velocity],
            dtype=np.float32,
        ),
    }
    return environment, observation


def rollout_dynamic_branch(record, actions):
    environment, initial = reset_dynamic_environment(
        record["state"], record_task(record), record["evaluation_seed"]
    )
    observations = {}
    states = {}
    counts = {}
    kinds = {}
    cumulative = 0
    try:
        for step, action in enumerate(actions, start=1):
            observation, _, _, info = environment.step(action)
            cumulative += int(info.get("n_contacts", 0))
            if step % FRAMESKIP == 0:
                horizon = step // FRAMESKIP
                if horizon in TARGET_STEPS:
                    observations[horizon] = {
                        "visual": np.asarray(observation["visual"]).copy(),
                        "proprio": np.asarray(observation["proprio"]).copy(),
                    }
                    states[horizon] = dynamic_state_from_environment(environment)
                    counts[horizon] = cumulative
                    kinds[horizon] = "contact" if cumulative > 0 else "free"
    finally:
        environment.close()
    if set(observations) != set(TARGET_STEPS):
        raise RuntimeError("dynamic rollout missed a target horizon")
    return initial, observations, states, counts, kinds


def exact_dynamic_restore_test(record):
    first, first_observation = reset_dynamic_environment(
        record["state"], record_task(record), record["evaluation_seed"]
    )
    first_state = dynamic_state_from_environment(first)
    second, second_observation = reset_dynamic_environment(
        record["state"], record_task(record), record["evaluation_seed"]
    )
    second_state = dynamic_state_from_environment(second)
    test_action = candidate_action_bank(record["state"])[1, 0]
    first.step(test_action)
    second.step(test_action)
    first_next = dynamic_state_from_environment(first)
    second_next = dynamic_state_from_environment(second)
    first.close()
    second.close()
    result = {
        "state_exact": bool(np.allclose(first_state, second_state, atol=1e-12, rtol=0)),
        "visual_exact": bool(
            np.array_equal(first_observation["visual"], second_observation["visual"])
        ),
        "proprio_exact": bool(
            np.array_equal(first_observation["proprio"], second_observation["proprio"])
        ),
        "one_step_continuation_exact": bool(
            np.allclose(first_next, second_next, atol=1e-12, rtol=0)
        ),
    }
    result["passed"] = bool(all(result.values()))
    if not result["passed"]:
        raise RuntimeError(f"full dynamic restore test failed: {result}")
    return result


def trajectory_path(spec):
    return TRUTH_DIR / f"trajectory_{int(spec['trajectory_id']):03d}.npz"


def realize_trajectory(spec):
    destination = trajectory_path(spec)
    if destination.exists():
        with np.load(destination) as payload:
            return [
                {
                    "record_id": int(spec["trajectory_id"]) * STATES_PER_TRAJECTORY + index,
                    "trajectory_id": int(spec["trajectory_id"]),
                    "time_index": index,
                    "physical_step": int(LONGITUDINAL_SAVE_STEPS[index]),
                    "task_id": int(spec["task_id"]),
                    "split": spec["split"],
                    "evaluation_seed": int(spec["seed"]),
                    "state": payload["states"][index].astype(np.float64),
                    "path_contacts": int(payload["cumulative_contacts"][index]),
                }
                for index in ACTIVE_TIME_INDICES
            ]
    environment, _ = reset_dynamic_environment(
        spec["initial_state"], record_task(spec), int(spec["seed"])
    )
    states = [dynamic_state_from_environment(environment)]
    contacts = [0]
    cumulative = 0
    try:
        for step, action in enumerate(spec["controls"], start=1):
            result = environment.step(action)
            if len(result) == 4:
                _, _, _, info = result
            elif len(result) == 5:
                _, _, _, _, info = result
            else:
                raise RuntimeError(f"unexpected PushT step result of length {len(result)}")
            cumulative += int(info.get("n_contacts", 0))
            if step in LONGITUDINAL_SAVE_STEPS[1:]:
                states.append(dynamic_state_from_environment(environment))
                contacts.append(cumulative)
    finally:
        environment.close()
    if len(states) != STATES_PER_TRAJECTORY:
        raise RuntimeError("longitudinal trajectory missed a frozen save point")
    atomic_npz(
        destination,
        trajectory_id=np.asarray(spec["trajectory_id"], dtype=np.int64),
        split=np.asarray(spec["split"]),
        states=np.stack(states).astype(np.float64),
        controls=np.asarray(spec["controls"], dtype=np.float64),
        save_steps=np.asarray(LONGITUDINAL_SAVE_STEPS, dtype=np.int64),
        cumulative_contacts=np.asarray(contacts, dtype=np.int64),
    )
    return realize_trajectory(spec)


def realize_records(specs):
    records = []
    for spec in specs:
        records.extend(realize_trajectory(spec))
    return records


RAW_ACTION_BASIS = temporal_action_basis(
    max(HORIZONS) * FRAMESKIP, ACTION_PROFILES
)


def candidate_action_bank(state):
    state = np.asarray(state, dtype=np.float64)
    if state.shape[-1] < 5:
        raise ValueError("candidate state is not a PushT state")
    baseline = np.zeros((max(HORIZONS) * FRAMESKIP, 2), dtype=np.float64)
    branches = [baseline]
    for column in range(ACTION_BASIS_DIM):
        delta = (
            ACTION_TANGENT_NORM
            * RAW_ACTION_BASIS[:, column].reshape(-1, 2)
        )
        branches.extend([baseline + delta, baseline - delta])
    actions = np.stack(branches)
    if actions.shape != (
        ACTIONS_PER_STATE,
        max(HORIZONS) * FRAMESKIP,
        2,
    ):
        raise RuntimeError(f"bad fixed action bank shape {actions.shape}")
    if np.max(np.abs(actions)) > 0.14:
        raise RuntimeError("fixed relative action perturbation exceeds safe magnitude")
    return actions.astype(np.float32)


def branch_path(record_id):
    return TRUTH_DIR / f"state_{int(record_id):04d}.npz"


def generate_truth(records):
    started = time.perf_counter()
    for record_index, record in enumerate(records):
        destination = branch_path(record["record_id"])
        if destination.exists():
            continue
        selected_actions = candidate_action_bank(record["state"])
        initials = []
        initial_proprios = []
        future_visual = []
        future_proprio = []
        endpoint_states = []
        interaction_counts = []
        interaction_types = []
        for action in selected_actions:
            initial, observations, states, counts, kinds = rollout_dynamic_branch(
                record, action
            )
            initials.append(initial["visual"])
            initial_proprios.append(initial["proprio"])
            future_visual.append([observations[h]["visual"] for h in HORIZONS])
            future_proprio.append([observations[h]["proprio"] for h in HORIZONS])
            endpoint_states.append([states[h] for h in HORIZONS])
            interaction_counts.append([counts[h] for h in HORIZONS])
            interaction_types.append([kinds[h] for h in HORIZONS])
        if not all(np.array_equal(initials[0], value) for value in initials[1:]):
            raise AssertionError("initial visual drift across action branches")
        if not all(
            np.array_equal(initial_proprios[0], value)
            for value in initial_proprios[1:]
        ):
            raise AssertionError("initial proprio drift across action branches")
        atomic_npz(
            destination,
            record_id=np.asarray(record["record_id"], dtype=np.int64),
            trajectory_id=np.asarray(record["trajectory_id"], dtype=np.int64),
            time_index=np.asarray(record["time_index"], dtype=np.int64),
            physical_step=np.asarray(record["physical_step"], dtype=np.int64),
            split=np.asarray(record["split"]),
            initial_state=np.asarray(record["state"], dtype=np.float64),
            initial_visual=np.asarray(initials[0], dtype=np.uint8),
            initial_proprio=np.asarray(initial_proprios[0], dtype=np.float32),
            selected_actions=selected_actions,
            future_visual=np.asarray(future_visual, dtype=np.uint8),
            future_proprio=np.asarray(future_proprio, dtype=np.float32),
            endpoint_states=np.asarray(endpoint_states, dtype=np.float32),
            interaction_counts=np.asarray(interaction_counts, dtype=np.int32),
            interaction_types=np.asarray(interaction_types),
        )
        write_json(
            OUT / f"{record['split']}_truth_progress.json",
            {
                "completed": record_index + 1,
                "total": len(records),
                "last_record_id": record["record_id"],
            },
        )
    TIMINGS[f"{records[0]['split']}_truth_seconds"] = (
        time.perf_counter() - started
    )


CONSTRUCTION_RECORDS = []
EVALUATION_RECORDS = []
ALL_RECORDS = []
RECORD_BY_ID = {}
if not PIPELINE_FAILED:
    try:
        REPO = configure_repo()
        CONSTRUCTION_RECORDS = realize_records(CONSTRUCTION_SPECS)
        restore_record = CONSTRUCTION_RECORDS[0]
        RESTORE_TEST = exact_dynamic_restore_test(restore_record)
        write_json(OUT / "restore_test.json", RESTORE_TEST)
        generate_truth(CONSTRUCTION_RECORDS)
        memory_report("construction_truth_complete")
    except Exception:
        record_failure("construction_truth_generation")

In [ ]:
# Fit and hash every construction-only reader before evaluation trajectories exist.


def target_path(record_id):
    return TARGET_DIR / f"state_{int(record_id):04d}.npz"


def encode_target_cache(records):
    started = time.perf_counter()
    for index, record in enumerate(records):
        destination = target_path(record["record_id"])
        if destination.exists():
            continue
        with np.load(branch_path(record["record_id"])) as truth:
            visual = truth["future_visual"]
            proprio = truth["future_proprio"]
        with torch.inference_mode():
            encoded = MODEL.encode(to_model_observation(visual, proprio))
        tokens = encoded["visual"][:, :, 0].reshape(
            ACTIONS_PER_STATE, len(HORIZONS), 256, encoded["visual"].shape[-1]
        ).detach().float().cpu().numpy()
        if tokens.shape != (13, 2, 256, 384):
            raise RuntimeError(f"unexpected target-token shape {tokens.shape}")
        atomic_npz(destination, true_tokens=tokens.astype(np.float32))
        write_json(OUT / f"{record['split']}_target_progress.json", {
            "completed": index + 1, "total": len(records), "last_record_id": record["record_id"]
        })
    TIMINGS[f"{records[0]['split']}_target_encoding_seconds"] = time.perf_counter() - started


def load_target_tokens(record_id):
    with np.load(target_path(record_id)) as payload:
        return payload["true_tokens"].astype(np.float32)


def state_model_inputs(record_id, horizon):
    with np.load(branch_path(record_id)) as truth:
        initial_visual = truth["initial_visual"]
        initial_proprio = truth["initial_proprio"]
        selected_actions = truth["selected_actions"]
    with torch.inference_mode():
        initial = MODEL.encode(to_model_observation(initial_visual, initial_proprio))
    initial = {name: value.detach() for name, value in initial.items()}
    actions = model_action_tensor(PREPROCESSOR, selected_actions, horizon)
    return initial, actions


rows, columns = np.meshgrid(np.arange(16), np.arange(16), indexing="ij")
PATCH_COORDINATES = np.stack([
    2.0 * (columns.reshape(-1) + 0.5) / 16.0 - 1.0,
    2.0 * (rows.reshape(-1) + 0.5) / 16.0 - 1.0,
], axis=1).astype(np.float64)
atomic_npz(DESIGN_DIR / "patch_coordinates.npz", coordinates=PATCH_COORDINATES)

GLOBAL_SCRAMBLE = np.random.default_rng(GLOBAL_SCRAMBLE_SEED).permutation(256)
atomic_npz(DESIGN_DIR / "feature_randomness.npz", global_scramble=GLOBAL_SCRAMBLE)


def moment_features(tokens, basis):
    values = np.asarray(tokens, dtype=np.float32)
    weights = np.asarray(basis, dtype=np.float32)
    return (np.einsum("npd,pk->nkd", values, weights, optimize=True) / 256.0).reshape(len(values), -1)


def gpu_countsketch(values, output_dim, seed, batch_size=64):
    array = np.asarray(values, dtype=np.float32)
    projector = CountSketchProjector(int(np.prod(array.shape[1:])), output_dim, seed)
    chunks = []
    with torch.inference_mode():
        for start in range(0, len(array), batch_size):
            tensor = torch.from_numpy(array[start : start + batch_size]).to("cuda")
            chunks.append(projector(tensor).cpu().numpy())
    return np.concatenate(chunks, axis=0)


def within_example_moments(tokens, metadata):
    result = np.empty((len(tokens), 1152), dtype=np.float32)
    for index, (trajectory_id, time_index, action_index, horizon) in enumerate(metadata):
        seed = np.random.SeedSequence([
            WITHIN_PERMUTATION_ROOT, trajectory_id, time_index, action_index, horizon
        ])
        permutation = np.random.default_rng(seed).permutation(256)
        basis = np.column_stack((np.ones(256), PATCH_COORDINATES[permutation]))
        result[index] = moment_features(tokens[index : index + 1], basis)[0]
    return result


def object_masks(image):
    red = image[..., 0].astype(np.int16)
    green = image[..., 1].astype(np.int16)
    blue = image[..., 2].astype(np.int16)
    agent = (blue - red > 80) & (blue - green > 30) & (blue > 180) & (green < 190)
    block = (blue - red > 20) & (green - red > 8) & (blue < 220) & ~agent
    return agent, block


def patch_fraction(mask):
    return mask.reshape(16, 14, 16, 14).mean(axis=(1, 3))


def mask_geometry(mask):
    weights = np.asarray(mask, dtype=np.float64)
    height, width = weights.shape
    yy, xx = np.meshgrid(
        (np.arange(height) + 0.5) / height,
        (np.arange(width) + 0.5) / width,
        indexing="ij",
    )
    total = max(float(weights.sum()), 1e-12)
    center_x, center_y = float(np.sum(weights * xx) / total), float(np.sum(weights * yy) / total)
    dx, dy = xx - center_x, yy - center_y
    moments = [
        np.sum(weights * dx**2) / total, np.sum(weights * dx * dy) / total,
        np.sum(weights * dy**2) / total, np.sum(weights * dx**3) / total,
        np.sum(weights * dx**2 * dy) / total, np.sum(weights * dx * dy**2) / total,
        np.sum(weights * dy**3) / total,
    ]
    return np.asarray([center_x, center_y, *moments], dtype=np.float32)


def render_oracle_features(images):
    rows = []
    for image in images:
        agent, block = object_masks(image)
        rows.append(np.concatenate((mask_geometry(patch_fraction(agent))[:2], mask_geometry(patch_fraction(block)))))
    return np.stack(rows)


def all_features(tokens, metadata, images, native_proprio):
    degree1_basis = np.column_stack((np.ones(256), PATCH_COORDINATES))
    x, y = PATCH_COORDINATES.T
    degree2_basis = np.column_stack((np.ones(256), x, y, x*x, x*y, y*y))
    scrambled_basis = np.column_stack((np.ones(256), PATCH_COORDINATES[GLOBAL_SCRAMBLE]))
    grid = tokens.reshape(len(tokens), 16, 16, 384)
    pools = np.concatenate([
        grid[:, :8, :8].mean((1, 2)), grid[:, :8, 8:].mean((1, 2)),
        grid[:, 8:, :8].mean((1, 2)), grid[:, 8:, 8:].mean((1, 2)),
    ], axis=1)
    features = {
        "degree1": moment_features(tokens, degree1_basis),
        "degree2": moment_features(tokens, degree2_basis),
        "channel_mean": tokens.mean(axis=1),
        "spatial_pool": gpu_countsketch(pools, 1152, SPATIAL_POOL_SEED),
        "global_scramble": moment_features(tokens, scrambled_basis),
        "within_permutation": within_example_moments(tokens, metadata),
        "render_oracle": render_oracle_features(images),
        "native_proprio": np.asarray(native_proprio, dtype=np.float32),
    }
    for seed in ORIGINAL_SKETCH_SEEDS:
        features[f"original192_{seed}"] = gpu_countsketch(tokens, 192, seed)
    for seed in MATCHED_SKETCH_SEEDS:
        features[f"matched1152_{seed}"] = gpu_countsketch(tokens, 1152, seed)
    return features


def collect_horizon(records, horizon):
    horizon_index = HORIZONS.index(horizon)
    token_rows, target_rows, group_rows, image_rows, native_rows, metadata = [], [], [], [], [], []
    for record in records:
        token_rows.append(load_target_tokens(record["record_id"])[:, horizon_index])
        with np.load(branch_path(record["record_id"])) as truth:
            endpoints = truth["endpoint_states"][:, horizon_index]
            image_rows.append(truth["future_visual"][:, horizon_index])
        target_rows.append(physical_reader_targets(endpoints))
        group_rows.append(np.full(ACTIONS_PER_STATE, record["trajectory_id"], dtype=np.int64))
        initial, actions = state_model_inputs(record["record_id"], horizon)
        with torch.inference_mode():
            _, predicted_proprio, _ = forward_with_carriers(initial, actions, horizon)
        native_rows.append(predicted_proprio.detach().float().cpu().numpy())
        metadata.extend([
            (record["trajectory_id"], record["time_index"], action_index, horizon)
            for action_index in range(ACTIONS_PER_STATE)
        ])
    return {
        "tokens": np.concatenate(token_rows),
        "targets": np.concatenate(target_rows),
        "groups": np.concatenate(group_rows),
        "images": np.concatenate(image_rows),
        "native_proprio": np.concatenate(native_rows),
        "metadata": metadata,
    }


def fit_ridge_dual(features, targets, ridge):
    x = np.asarray(features, dtype=np.float64)
    y = np.asarray(targets, dtype=np.float64)
    feature_mean, feature_scale = x.mean(0), x.std(0)
    feature_scale = np.where(feature_scale > 1e-8, feature_scale, 1.0)
    xs = (x - feature_mean) / feature_scale
    intercept = y.mean(0)
    yc = y - intercept
    gram = xs @ xs.T
    eigenvalues, eigenvectors = np.linalg.eigh((gram + gram.T) / 2.0)
    eigenvalues = np.maximum(eigenvalues, 0.0)
    alpha = eigenvectors @ ((eigenvectors.T @ yc) / (eigenvalues[:, None] + float(ridge)))
    coefficient = xs.T @ alpha
    return {
        "feature_mean": feature_mean, "feature_scale": feature_scale,
        "intercept": intercept, "coefficient": coefficient, "ridge": float(ridge),
    }


def predict_model(model, features):
    x = np.asarray(features, dtype=np.float64)
    xs = (x - model["feature_mean"]) / model["feature_scale"]
    return xs @ model["coefficient"] + model["intercept"]


def grouped_cv_dual(features, targets, groups, ridges, prefer_larger=True):
    groups = np.asarray(groups)
    totals = {float(ridge): [] for ridge in ridges}
    rows = []
    for held_out in np.unique(groups):
        train = groups != held_out
        test = ~train
        for ridge in ridges:
            model = fit_ridge_dual(features[train], targets[train], ridge)
            mse = float(np.mean((predict_model(model, features[test]) - targets[test]) ** 2))
            totals[float(ridge)].append(mse)
            rows.append({"ridge": float(ridge), "held_out_group": int(held_out), "mse": mse})
    means = {ridge: float(np.mean(values)) for ridge, values in totals.items()}
    for ridge, mse in means.items():
        rows.append({"ridge": ridge, "held_out_group": "mean", "mse": mse})
    selected = min(means, key=lambda ridge: (means[ridge], -ridge if prefer_larger else ridge))
    return fit_ridge_dual(features, targets, selected), rows


def save_model(horizon, name, model):
    destination = MODEL_DIR / f"h{horizon}_{name}.npz"
    atomic_npz(destination, **model)
    return {"path": str(destination), "sha256": sha256_file(destination)}


def load_model(path):
    with np.load(path) as payload:
        return {name: payload[name].copy() for name in payload.files}


def load_frozen_bundle(freeze):
    models = {str(h): {} for h in HORIZONS}
    for horizon, entries in freeze["models"].items():
        for name, item in entries.items():
            if sha256_file(item["path"]) != item["sha256"]:
                raise RuntimeError("frozen reader hash mismatch")
            models[horizon][name] = load_model(item["path"])
    normalizers = {
        horizon: {
            "mean": np.asarray(value["mean"], dtype=np.float64),
            "scale": np.asarray(value["scale"], dtype=np.float64),
        }
        for horizon, value in freeze["target_normalizers"].items()
    }
    return models, normalizers


if not PIPELINE_FAILED:
    try:
        MODEL, PREPROCESSOR, PREDICTOR, PREDICTOR_BLOCK_MODULES = load_frozen_model()
        encode_target_cache(CONSTRUCTION_RECORDS)
        with np.load(branch_path(CONSTRUCTION_RECORDS[0]["record_id"])) as truth:
            visual, proprio = truth["future_visual"][:1], truth["future_proprio"][:1]
        with torch.inference_mode():
            first = MODEL.encode(to_model_observation(visual, proprio))["visual"]
            second = MODEL.encode(to_model_observation(visual, proprio))["visual"]
        repeat_error = float(torch.max(torch.abs(first - second)).cpu())
        write_json(ANALYSIS_DIR / "determinism_check.json", {"max_abs_error": repeat_error, "passed": repeat_error == 0.0})
        if repeat_error != 0.0:
            raise RuntimeError("target encoding is not exactly deterministic")

        freeze_path = ANALYSIS_DIR / "reader_freeze.json"
        if freeze_path.exists():
            freeze = json.loads(freeze_path.read_text())
            if freeze["protocol_sha256"] != PROTOCOL_SHA256:
                raise RuntimeError("existing reader freeze belongs to another protocol")
            FROZEN_MODELS, TARGET_NORMALIZERS = load_frozen_bundle(freeze)
        else:
            fit_started = time.perf_counter()
            FROZEN_MODELS, TARGET_NORMALIZERS, freeze_models, cv_rows = {}, {}, {}, []
            for horizon in HORIZONS:
                data = collect_horizon(CONSTRUCTION_RECORDS, horizon)
                target_mean, target_scale = data["targets"].mean(0), data["targets"].std(0)
                if np.any(target_scale <= 1e-5):
                    raise RuntimeError(f"degenerate construction target at horizon {horizon}")
                standardized_target = (data["targets"] - target_mean) / target_scale
                features = all_features(
                    data["tokens"], data["metadata"], data["images"], data["native_proprio"]
                )
                models, model_files = {}, {}
                for name, values in features.items():
                    old_exact = name.startswith("original192_")
                    grid = ORIGINAL_RIDGE_GRID if old_exact else RIDGE_GRID
                    model, rows = grouped_cv_dual(
                        values, standardized_target, data["groups"], grid,
                        prefer_larger=not old_exact,
                    )
                    models[name] = model
                    model_files[name] = save_model(horizon, name, model)
                    cv_rows.extend({"horizon": horizon, "reader": name, **row} for row in rows)
                FROZEN_MODELS[str(horizon)] = models
                freeze_models[str(horizon)] = model_files
                TARGET_NORMALIZERS[str(horizon)] = {"mean": target_mean, "scale": target_scale}
                del features, data
                gc.collect()
            write_csv(ANALYSIS_DIR / "construction_cv.csv", cv_rows)
            freeze = {
                "protocol_id": PROTOCOL_ID,
                "protocol_sha256": PROTOCOL_SHA256,
                "frozen_unix": time.time(),
                "construction_trajectories": CONSTRUCTION_TRAJECTORIES,
                "evaluation_trajectories_seen": [],
                "models": freeze_models,
                "target_normalizers": {
                    horizon: {"mean": value["mean"].tolist(), "scale": value["scale"].tolist()}
                    for horizon, value in TARGET_NORMALIZERS.items()
                },
                "frozen_before_evaluation": True,
            }
            write_json(freeze_path, freeze)
            TIMINGS["reader_fit_seconds"] = time.perf_counter() - fit_started

        construction_active = TIMINGS.get("construction_truth_seconds", 0.0) + TIMINGS.get("construction_target_encoding_seconds", 0.0)
        projected_minutes = (3.0 * construction_active + TIMINGS.get("reader_fit_seconds", 0.0) + 300.0) / 60.0
        current_bytes = sum(path.stat().st_size for path in OUT.rglob("*") if path.is_file())
        projected_raw_gb = 3.0 * current_bytes / 1e9
        memory = memory_report("reader_frozen")
        guards = {
            "projected_wall_minutes": projected_minutes,
            "projected_raw_gb": projected_raw_gb,
            "gpu_peak_allocated_gib": memory["gpu_peak_allocated_gib"],
            "wall_pass": projected_minutes <= MAX_PROJECTED_WALL_MINUTES,
            "storage_pass": projected_raw_gb <= MAX_PROJECTED_RAW_GB,
            "memory_pass": memory["gpu_peak_allocated_gib"] <= MAX_PEAK_GPU_GB,
        }
        write_json(ANALYSIS_DIR / "runtime_guards.json", guards)
        if not all(guards[key] for key in ("wall_pass", "storage_pass", "memory_pass")):
            raise RuntimeError(f"runtime guard failed before evaluation: {guards}")
        print("All readers frozen; evaluation remains unopened.")
    except Exception:
        record_failure("construction_reader_freeze")

In [ ]:
# Verify the delivered code prefix, then open evaluation exactly once and apply the frozen gate.

GATE_SOURCE = '#!/usr/bin/env python3\n"""Deterministic decision gate for stage16-coordinate-moment-confirmation-v1.\n\nThis script does not fit a reader. It consumes trajectory-aggregated squared errors\nproduced by a source-bound evaluator after all readers have been frozen.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\nfrom statistics import median\nfrom typing import Any\n\n\nPROTOCOL_ID = "stage16-coordinate-moment-confirmation-v1"\nHORIZONS = ("1", "3")\nLABELS = (\n    "agent_x",\n    "agent_y",\n    "block_x",\n    "block_y",\n    "block_sin",\n    "block_cos",\n)\nSPATIAL_INDICES = (0, 1, 2, 3)\nEVALUATION_TRAJECTORIES = (\n    101,\n    102,\n    104,\n    105,\n    107,\n    108,\n    110,\n    111,\n    113,\n    114,\n    116,\n    117,\n    119,\n    120,\n    122,\n    123,\n)\nMIN_MEDIAN_SKILL = 0.30\nMIN_SPATIAL_SKILL = 0.20\nMIN_LOO_MEDIAN_SKILL = 0.20\nMIN_LOO_SPATIAL_SKILL = 0.0\nMIN_JOINT_POSITIVE_TRAJECTORIES = 12\nMIN_MEDIAN_RELATIVE_REDUCTION = 0.10\nMIN_POOLED_SD_RATIO = 0.25\nMIN_WITHIN_VARIANCE_TRAJECTORIES = 8\n\n\nclass ProtocolError(ValueError):\n    """Raised when an input violates the frozen gate schema."""\n\n\ndef _vector(value: Any, where: str) -> list[float]:\n    if not isinstance(value, list) or len(value) != len(LABELS):\n        raise ProtocolError(f"{where} must be a six-element list")\n    result = [float(item) for item in value]\n    if any(not math.isfinite(item) or item < 0.0 for item in result):\n        raise ProtocolError(f"{where} must contain finite nonnegative MSE values")\n    return result\n\n\ndef _mean_vectors(vectors: list[list[float]]) -> list[float]:\n    if not vectors:\n        raise ProtocolError("cannot aggregate an empty trajectory set")\n    return [sum(row[index] for row in vectors) / len(vectors) for index in range(6)]\n\n\ndef _skills(rows: list[dict[str, Any]]) -> list[float]:\n    baseline = _mean_vectors([row["baseline_mse"] for row in rows])\n    primary = _mean_vectors([row["primary_mse"] for row in rows])\n    if any(value <= 0.0 for value in baseline):\n        raise ProtocolError("construction-mean baseline MSE must be positive for every target")\n    return [1.0 - error / null for error, null in zip(primary, baseline)]\n\n\ndef _relative_reduction(primary: float, control: float) -> float:\n    if control > 0.0:\n        return (control - primary) / control\n    return 0.0 if primary == 0.0 else float("-inf")\n\n\ndef _all_true(value: Any) -> bool:\n    if isinstance(value, bool):\n        return value\n    if isinstance(value, dict):\n        return bool(value) and all(_all_true(item) for item in value.values())\n    if isinstance(value, list):\n        return bool(value) and all(_all_true(item) for item in value)\n    raise ProtocolError("integrity_checks may contain only nested booleans")\n\n\ndef compute_gate(payload: dict[str, Any]) -> dict[str, Any]:\n    """Compute the one frozen Stage 16 decision from an evaluator payload."""\n\n    if payload.get("protocol_id") != PROTOCOL_ID:\n        raise ProtocolError(f"protocol_id must equal {PROTOCOL_ID}")\n\n    integrity = payload.get("integrity_checks")\n    if not isinstance(integrity, dict):\n        raise ProtocolError("integrity_checks must be an object")\n    integrity_pass = _all_true(integrity)\n\n    variance = payload.get("variance_checks")\n    if not isinstance(variance, dict) or set(variance) != set(HORIZONS):\n        raise ProtocolError("variance_checks must contain exactly horizons 1 and 3")\n    variance_details: dict[str, Any] = {}\n    variance_pass = True\n    for horizon in HORIZONS:\n        horizon_values = variance[horizon]\n        if not isinstance(horizon_values, dict) or set(horizon_values) != set(LABELS):\n            raise ProtocolError(f"variance_checks[{horizon}] must contain all six labels")\n        variance_details[horizon] = {}\n        for label in LABELS:\n            item = horizon_values[label]\n            try:\n                pooled_sd_ratio = float(item["pooled_sd_ratio"])\n                eligible_count = int(item["within_sd_ratio_ge_0_10_count"])\n            except (KeyError, TypeError, ValueError) as error:\n                raise ProtocolError(f"bad variance entry for horizon {horizon}, {label}") from error\n            if not math.isfinite(pooled_sd_ratio) or pooled_sd_ratio < 0.0:\n                raise ProtocolError("pooled_sd_ratio must be finite and nonnegative")\n            passed = (\n                pooled_sd_ratio >= MIN_POOLED_SD_RATIO\n                and eligible_count >= MIN_WITHIN_VARIANCE_TRAJECTORIES\n            )\n            variance_details[horizon][label] = {\n                "pooled_sd_ratio": pooled_sd_ratio,\n                "within_sd_ratio_ge_0_10_count": eligible_count,\n                "passed": passed,\n            }\n            variance_pass = variance_pass and passed\n\n    raw_horizons = payload.get("horizons")\n    if not isinstance(raw_horizons, dict) or set(raw_horizons) != set(HORIZONS):\n        raise ProtocolError("horizons must contain exactly 1 and 3")\n\n    parsed: dict[str, dict[int, dict[str, Any]]] = {}\n    horizon_results: dict[str, Any] = {}\n    advantages: dict[int, list[float]] = {value: [] for value in EVALUATION_TRAJECTORIES}\n    relative_reductions: dict[str, list[float]] = {value: [] for value in HORIZONS}\n\n    for horizon in HORIZONS:\n        trajectories = raw_horizons[horizon].get("trajectories")\n        if not isinstance(trajectories, dict):\n            raise ProtocolError(f"horizons[{horizon}].trajectories must be an object")\n        try:\n            ids = tuple(sorted(int(value) for value in trajectories))\n        except ValueError as error:\n            raise ProtocolError("trajectory keys must be integer strings") from error\n        if ids != EVALUATION_TRAJECTORIES:\n            raise ProtocolError(\n                f"horizon {horizon} evaluation IDs differ from the frozen 16-trajectory split"\n            )\n\n        parsed[horizon] = {}\n        for trajectory_id in EVALUATION_TRAJECTORIES:\n            item = trajectories[str(trajectory_id)]\n            baseline = _vector(item.get("baseline_mse"), f"{horizon}/{trajectory_id}/baseline")\n            primary = _vector(item.get("primary_mse"), f"{horizon}/{trajectory_id}/primary")\n            sketches = item.get("matched_sketch_mse")\n            if not isinstance(sketches, list) or len(sketches) != 3:\n                raise ProtocolError(\n                    f"{horizon}/{trajectory_id}/matched_sketch_mse must have three replicates"\n                )\n            sketch_vectors = [\n                _vector(value, f"{horizon}/{trajectory_id}/matched_sketch[{index}]")\n                for index, value in enumerate(sketches)\n            ]\n            spatial_pool = _vector(\n                item.get("spatial_pool_mse"), f"{horizon}/{trajectory_id}/spatial_pool"\n            )\n            global_scramble = _vector(\n                item.get("global_scramble_mse"),\n                f"{horizon}/{trajectory_id}/global_scramble",\n            )\n            primary_mean = sum(primary) / 6.0\n            control_means = [sum(value) / 6.0 for value in sketch_vectors]\n            control_means.extend((sum(spatial_pool) / 6.0, sum(global_scramble) / 6.0))\n            best_control = min(control_means)\n            advantages[trajectory_id].append(best_control - primary_mean)\n            relative_reductions[horizon].append(\n                _relative_reduction(primary_mean, best_control)\n            )\n            parsed[horizon][trajectory_id] = {\n                "baseline_mse": baseline,\n                "primary_mse": primary,\n                "best_control_mean_mse": best_control,\n                "primary_mean_mse": primary_mean,\n            }\n\n        all_rows = list(parsed[horizon].values())\n        skills = _skills(all_rows)\n        absolute_pass = (\n            median(skills) >= MIN_MEDIAN_SKILL\n            and min(skills[index] for index in SPATIAL_INDICES) >= MIN_SPATIAL_SKILL\n        )\n        loo_rows = []\n        loo_pass = True\n        for deleted in EVALUATION_TRAJECTORIES:\n            subset = [row for key, row in parsed[horizon].items() if key != deleted]\n            loo_skills = _skills(subset)\n            passed = (\n                median(loo_skills) >= MIN_LOO_MEDIAN_SKILL\n                and min(loo_skills[index] for index in SPATIAL_INDICES)\n                > MIN_LOO_SPATIAL_SKILL\n            )\n            loo_pass = loo_pass and passed\n            loo_rows.append(\n                {\n                    "deleted_trajectory": deleted,\n                    "median_skill": median(loo_skills),\n                    "minimum_spatial_skill": min(\n                        loo_skills[index] for index in SPATIAL_INDICES\n                    ),\n                    "passed": passed,\n                }\n            )\n        horizon_results[horizon] = {\n            "skill": dict(zip(LABELS, skills)),\n            "median_skill": median(skills),\n            "minimum_spatial_skill": min(skills[index] for index in SPATIAL_INDICES),\n            "absolute_pass": absolute_pass,\n            "leave_one_trajectory_out": loo_rows,\n            "loo_pass": loo_pass,\n        }\n\n    joint_advantage = {\n        str(trajectory_id): min(values)\n        for trajectory_id, values in advantages.items()\n    }\n    joint_positive_count = sum(value > 0.0 for value in joint_advantage.values())\n    specificity_pass = joint_positive_count >= MIN_JOINT_POSITIVE_TRAJECTORIES\n    relative_medians = {\n        horizon: median(values) for horizon, values in relative_reductions.items()\n    }\n    relative_reduction_pass = all(\n        value >= MIN_MEDIAN_RELATIVE_REDUCTION for value in relative_medians.values()\n    )\n    absolute_pass = all(value["absolute_pass"] for value in horizon_results.values())\n    loo_pass = all(value["loo_pass"] for value in horizon_results.values())\n\n    components = {\n        "absolute_skill": absolute_pass,\n        "leave_one_trajectory_out": loo_pass,\n        "matched_control_specificity": specificity_pass,\n        "relative_error_reduction": relative_reduction_pass,\n        "target_variance": variance_pass,\n        "integrity": integrity_pass,\n    }\n    failure_reasons = [name for name, passed in components.items() if not passed]\n    return {\n        "protocol_id": PROTOCOL_ID,\n        "decision": "PASS" if not failure_reasons else "FAIL",\n        "passed": not failure_reasons,\n        "components": components,\n        "horizon_results": horizon_results,\n        "joint_control_advantage": {\n            "by_trajectory": joint_advantage,\n            "positive_count": joint_positive_count,\n            "required_positive_count": MIN_JOINT_POSITIVE_TRAJECTORIES,\n            "exact_one_sided_sign_p_at_boundary": 0.0384063720703125,\n        },\n        "median_relative_error_reduction": relative_medians,\n        "variance": variance_details,\n        "failure_reasons": failure_reasons,\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("metrics", type=Path, help="source-bound evaluator JSON")\n    parser.add_argument("--output", type=Path, help="optional decision JSON path")\n    arguments = parser.parse_args()\n    payload = json.loads(arguments.metrics.read_text(encoding="utf-8"))\n    result = compute_gate(payload)\n    rendered = json.dumps(result, indent=2, sort_keys=True) + "\\n"\n    if arguments.output:\n        arguments.output.write_text(rendered, encoding="utf-8")\n    else:\n        print(rendered, end="")\n\n\nif __name__ == "__main__":\n    main()\n'


def evaluation_metrics(records, horizon, models, normalizer):
    data = collect_horizon(records, horizon)
    target = (data["targets"] - normalizer["mean"]) / normalizer["scale"]
    features = all_features(
        data["tokens"], data["metadata"], data["images"], data["native_proprio"]
    )
    predictions = {name: predict_model(models[name], values) for name, values in features.items()}
    predictions["original192_ensemble"] = np.mean([
        predictions[f"original192_{seed}"] for seed in ORIGINAL_SKETCH_SEEDS
    ], axis=0)
    descriptive = {
        name: dict(zip(READER_LABELS, r2_per_output(target, value).tolist()))
        for name, value in predictions.items()
    }
    trajectory_rows = {}
    for trajectory_id in EVALUATION_TRAJECTORIES:
        mask = data["groups"] == trajectory_id
        trajectory_rows[str(trajectory_id)] = {
            "baseline_mse": np.mean(target[mask] ** 2, axis=0).tolist(),
            "primary_mse": np.mean((predictions["degree1"][mask] - target[mask]) ** 2, axis=0).tolist(),
            "matched_sketch_mse": [
                np.mean((predictions[f"matched1152_{seed}"][mask] - target[mask]) ** 2, axis=0).tolist()
                for seed in MATCHED_SKETCH_SEEDS
            ],
            "spatial_pool_mse": np.mean((predictions["spatial_pool"][mask] - target[mask]) ** 2, axis=0).tolist(),
            "global_scramble_mse": np.mean((predictions["global_scramble"][mask] - target[mask]) ** 2, axis=0).tolist(),
        }
    variance = {}
    for index, label in enumerate(READER_LABELS):
        pooled_sd = float(np.std(target[:, index]))
        within_count = sum(
            float(np.std(target[data["groups"] == trajectory_id, index])) >= MIN_WITHIN_SD_RATIO
            for trajectory_id in EVALUATION_TRAJECTORIES
        )
        variance[label] = {
            "pooled_sd_ratio": pooled_sd,
            "within_sd_ratio_ge_0_10_count": int(within_count),
        }
    return trajectory_rows, variance, descriptive


if not PIPELINE_FAILED:
    try:
        if hashlib.sha256(GATE_SOURCE.encode()).hexdigest() != GATE_SCRIPT_SHA256:
            raise RuntimeError("embedded decision-gate source hash mismatch")
        SOURCE_IDENTITY = verify_execution_prefix(PRE_EVALUATION_CODE_CELL_COUNT)
        freeze_path = ANALYSIS_DIR / "reader_freeze.json"
        freeze_sha = sha256_file(freeze_path)
        authorization_path = ANALYSIS_DIR / "evaluation_authorization.json"
        if authorization_path.exists():
            authorization = json.loads(authorization_path.read_text())
            if authorization["reader_freeze_sha256"] != freeze_sha:
                raise RuntimeError("reader freeze changed after evaluation authorization")
        else:
            authorization = {
                "protocol_id": PROTOCOL_ID,
                "reader_freeze_sha256": freeze_sha,
                "source_identity": SOURCE_IDENTITY,
                "authorized_unix": time.time(),
            }
            write_json(authorization_path, authorization)

        EVALUATION_RECORDS = realize_records(EVALUATION_SPECS)
        generate_truth(EVALUATION_RECORDS)
        encode_target_cache(EVALUATION_RECORDS)
        horizon_payload, variance_payload, descriptive_payload = {}, {}, {}
        for horizon in HORIZONS:
            rows, variance, descriptive = evaluation_metrics(
                EVALUATION_RECORDS, horizon, FROZEN_MODELS[str(horizon)], TARGET_NORMALIZERS[str(horizon)]
            )
            horizon_payload[str(horizon)] = {"trajectories": rows}
            variance_payload[str(horizon)] = variance
            descriptive_payload[str(horizon)] = descriptive

        all_specs_complete = all(trajectory_path(spec).exists() for spec in ALL_TRAJECTORY_SPECS)
        all_state_shards = all(
            branch_path(trajectory_id * STATES_PER_TRAJECTORY + time_index).exists()
            and target_path(trajectory_id * STATES_PER_TRAJECTORY + time_index).exists()
            for trajectory_id in CONSTRUCTION_TRAJECTORIES + EVALUATION_TRAJECTORIES
            for time_index in range(STATES_PER_TRAJECTORY)
        )
        evaluator_payload = {
            "protocol_id": PROTOCOL_ID,
            "integrity_checks": {
                "source_bound": SOURCE_IDENTITY["confirmation_eligible"],
                "protocol_hash_match": digest_json_runtime(FROZEN_PROTOCOL) == PROTOCOL_SHA256,
                "all_24_trajectories_complete": bool(all_specs_complete and all_state_shards),
                "reader_frozen_before_evaluation": authorization["reader_freeze_sha256"] == freeze_sha,
                "forbidden_trajectory_ids_absent": not bool(set(range(8)) & set(CONSTRUCTION_TRAJECTORIES + EVALUATION_TRAJECTORIES)),
            },
            "variance_checks": variance_payload,
            "horizons": horizon_payload,
        }
        write_json(EVIDENCE_DIR / "evaluator_metrics.json", evaluator_payload)
        write_json(EVIDENCE_DIR / "descriptive_r2.json", descriptive_payload)
        gate_namespace = {"__name__": "stage16_reader_gate_embedded"}
        exec(GATE_SOURCE, gate_namespace)
        DECISION = gate_namespace["compute_gate"](evaluator_payload)
        write_json(OUT / "stage16_decision.json", DECISION)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
        for axis, horizon in zip(axes, ("1", "3")):
            values = [DECISION["horizon_results"][horizon]["skill"][label] for label in READER_LABELS]
            axis.bar(np.arange(6), values)
            axis.axhline(0.20, color="black", linestyle="--", linewidth=1)
            axis.set_xticks(np.arange(6), READER_LABELS, rotation=45, ha="right")
            axis.set_title(f"Horizon {horizon} degree-1 skill")
        fig.tight_layout()
        fig.savefig(PLOT_DIR / "primary_skill.png", dpi=160)
        plt.show()
        memory_report("evaluation_complete")
        print(json.dumps(DECISION, indent=2))
    except Exception:
        record_failure("sealed_evaluation_and_gate")

In [ ]:
# Package the complete evidence bundle. A scientific FAIL is a completed run, not a pipeline error.

if PIPELINE_FAILED:
    FINAL = {
        "protocol_id": PROTOCOL_ID,
        "decision": "PIPELINE_FAILURE",
        "failure_message": FAILURE_MESSAGE,
    }
else:
    FINAL = {
        "protocol_id": PROTOCOL_ID,
        "decision": DECISION["decision"],
        "passed": DECISION["passed"],
        "claim_boundary": FROZEN_PROTOCOL["claim_boundary"],
        "next_action": (
            "Authorize only the prespecified narrow causal pilot"
            if DECISION["passed"]
            else "Stop coordinate operators at this checkpoint"
        ),
    }
    (OUT / "FAILURE_TRACE.txt").write_text("NONE\n")

write_json(OUT / "final_summary.json", FINAL)
write_json(OUT / "timings.json", TIMINGS)
manifest = manifest_rows(OUT)
write_json(OUT / "full_manifest.json", {
    "protocol_id": PROTOCOL_ID,
    "file_count": len(manifest),
    "total_bytes": sum(row["size_bytes"] for row in manifest),
    "files": manifest,
})
archive_base = Path("/content") / f"stage16_reader_{{RUN_SIGNATURE[:12]}}"
archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUT))
print(json.dumps(FINAL, indent=2))
print(f"Evidence directory: {OUT}")
print(f"Download archive: {archive}")
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception as error:
        print(f"Automatic download unavailable: {error}")